# COMSYS Hackathon 7 — Handwritten Multiscript OCR

## Team Information

**Team Name:** Hitsturt

**Team Members:**
- Debanjan Sarkar
- Madhurima Das
- Samman Das
- Ujan Mahapatra

---

## Submission Overview

This notebook contains the complete implementation of our solution for the COMSYS Hackathon 7 Handwritten Multiscript OCR Challenge.

The original project was developed as a modular repository containing separate training, inference, evaluation, configuration, and utility modules. As required by the competition submission format, the repository has been consolidated into a single executable Jupyter Notebook while preserving the original pipeline structure and logic.

The notebook reproduces the complete workflow:

1. Dataset preparation
2. Character detection
3. Character recognition
4. Model ensembling
5. Evaluation
6. Submission generation

---

## Important Execution Notes

### Repository-to-Notebook Conversion

This notebook was generated by merging a previously modular codebase into a single-file format for submission purposes.

As a result:

- Some file paths may require adjustment depending on the execution environment.
- Checkpoint locations should be verified before running training or inference cells.
- Dataset root paths should be updated to match the local machine or competition environment.
- Output directories may be modified as needed without affecting the pipeline logic.

The underlying implementation, model architectures, hyperparameters, and inference procedures remain unchanged from the original repository.

---

### Environment Verification

Before executing the notebook, please verify:

- Dataset directories exist and are correctly referenced.
- Detector checkpoint paths are valid.
- Recognizer checkpoint paths are valid.
- Required Python packages are installed.
- CUDA/GPU availability is correctly detected if GPU execution is intended.

---

### Reproducibility

All major configuration values, training parameters, random seeds, augmentation settings, ensemble settings, and inference thresholds are defined explicitly within this notebook.

The implementation is intended to reproduce the same pipeline behaviour as the original repository while remaining self-contained for evaluation purposes.

---

## Acknowledgements

This notebook reflects the team's experimental work, model training, hyperparameter tuning, validation studies, and competition-specific optimizations performed throughout the hackathon.

AI-assisted development tools were used during implementation and documentation, while all model selection, experimentation, evaluation, and final design decisions were made within the context of the competition workflow.

## 1. Title and Problem Statement

This notebook is a single, self-contained reproduction of a two-stage OCR system
built for **COMSYS Hackathon 7**. The task is to read handwritten document pages
that mix **Latin** and **Bengali** scripts and, for every character on a page,
predict both:

1. a **bounding box** (`x1, y1, x2, y2` in pixels), and
2. a **Unicode label string** (e.g. `U+0065`, or a multi-codepoint conjunct such
   as `U+09AE+U+09CD+U+09AC`).

There are **167 distinct label strings** across the annotated corpus. The
competition scores each predicted character against the ground truth with a
Hungarian-matched, IoU-gated metric:

```
char_score = 0.20 · script_score      (same Unicode script family?)
           + 0.40 · unicode_score      (exact label match?)
           + 0.40 · iou_score          (IoU of the matched box pair)
```

A match is only valid when `IoU ≥ 0.50`. **Missing** ground-truth boxes score 0
on all three terms; **extra** predicted boxes are ignored. This asymmetry is the
single most important fact about the problem: the metric punishes misses but is
free with false positives, which is why the detector is tuned for **maximum
recall** rather than precision.



## 2. Solution Overview

The system is a classic **detect-then-recognise** OCR pipeline. Detection and
recognition are decoupled so each can be optimised independently: the detector is
single-class (it only finds "where is a character"), and a separate CNN answers
"which character is it".

```
                          ┌─────────────────────────────────────────────┐
   page image  ─────────▶ │  STAGE 1 — DETECTION (YOLO11 ensemble)       │
   (full page)            │                                             │
                          │   YOLO11m (main)  ┐                          │
                          │   YOLO11s fold0   │  predict(augment=True)   │
                          │   YOLO11s fold1   ├─▶ concat boxes ─▶ NMS@0.85│
                          │   YOLO11s fold2   ┘     (merge duplicates)    │
                          └───────────────┬─────────────────────────────┘
                                          │ N boxes (xyxy, pixels)
                                          ▼
                          ┌─────────────────────────────────────────────┐
                          │  CROP EXTRACTION                            │
                          │  aspect-preserving pad (8% margin) → square  │
                          │  → resize 128, white background              │
                          └───────────────┬─────────────────────────────┘
                                          │ N crops
                                          ▼
                          ┌─────────────────────────────────────────────┐
                          │  STAGE 2 — RECOGNITION (ConvNeXt ensemble)   │
                          │                                             │
                          │   ConvNeXt-Small fold0 ┐                     │
                          │   ConvNeXt-Small fold1 ├─▶ mean(logits) ─▶   │
                          │   ConvNeXt-Small fold2 ┘   softmax → argmax  │
                          └───────────────┬─────────────────────────────┘
                                          │ N (label, confidence)
                                          ▼
                          ┌─────────────────────────────────────────────┐
                          │  SUBMISSION ASSEMBLY                        │
                          │  per page: majority-script flag + one        │
                          │  {unicode_value,bbox} record per box         │
                          └─────────────────────────────────────────────┘
```

**Why this architecture.**

- **Two stages, not one.** With only 17 annotated pages but ~6,700 characters,
  there is far more signal for a *classifier* (thousands of crop samples) than
  for a *multi-class detector* (which would have to learn 167 categories from a
  handful of pages). Splitting the problem lets the recogniser use heavy crop-level
  augmentation and class balancing.
- **Detector ensemble for recall.** Four detectors with test-time augmentation are
  unioned and then de-duplicated with a deliberately *high* NMS IoU (0.85), so
  near-duplicate boxes from different models survive unless they almost perfectly
  overlap. Combined with a very low confidence threshold this maximises the chance
  every true character is covered by at least one box.
- **Recogniser fold ensemble.** Averaging logits across three cross-validation
  folds reduces variance on the long tail of rare conjuncts.



## 3. Environment Setup

All imports, logging, the global RNG seed, and the central configuration live
here..


In [1]:
import os
import sys
import json
import math
import random
import shutil
import logging
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from collections import Counter
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import cv2
import yaml
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold, KFold
from scipy.optimize import linear_sum_assignment

# timm (recogniser backbones) and ultralytics (YOLO) are imported lazily inside
# the functions/classes that need them, exactly as in the original modules.

print("Imports OK | torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

c:\Users\DEBANJAN\anaconda3\envs\comsys_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK | torch 2.11.0+cu128 | CUDA available: True


### 3.1 Logging

Centralised colour logger. Every section creates a named logger so log lines carry the originating stage, and one shared file handler captures the whole run.


In [2]:
import logging
import sys
from pathlib import Path
from datetime import datetime
from typing import Optional


_COLOURS = {
    "DEBUG":    "\033[36m",   # cyan
    "INFO":     "\033[32m",   # green
    "WARNING":  "\033[33m",   # yellow
    "ERROR":    "\033[31m",   # red
    "CRITICAL": "\033[35m",   # magenta
}
_RESET = "\033[0m"

class _ColourFormatter(logging.Formatter):
    """Add ANSI colour to the level-name portion of the log record."""

    def format(self, record: logging.LogRecord) -> str:
        colour = _COLOURS.get(record.levelname, "")
        record.levelname = f"{colour}{record.levelname:<8}{_RESET}"
        return super().format(record)


_file_handler: Optional[logging.FileHandler] = None
_log_file_path: Optional[Path] = None

def setup_logging(
    log_dir: str = "logs",
    run_name: Optional[str] = None,
    level: int = logging.DEBUG,
) -> Path:
    """
    Initialise the shared file handler.  Call once per run (e.g. from main()).

    Parameters
    ----------
    log_dir   : Directory where log files are saved.
    run_name  : Optional prefix for the log filename.  Defaults to timestamp.
    level     : Minimum logging level written to the file.

    Returns
    -------
    Path to the created log file.
    """
    global _file_handler, _log_file_path

    log_dir_path = Path(log_dir)
    log_dir_path.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    prefix = f"{run_name}_" if run_name else ""
    log_file = log_dir_path / f"{prefix}{timestamp}.log"

    _log_file_path = log_file
    _file_handler = logging.FileHandler(log_file, encoding="utf-8")
    _file_handler.setLevel(level)
    _file_handler.setFormatter(
        logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S",
        )
    )

    # Attach to root so all child loggers inherit it
    root = logging.getLogger()
    root.setLevel(level)
    root.addHandler(_file_handler)

    return log_file

def get_logger(name: str, level: int = logging.DEBUG) -> logging.Logger:
    """
    Return a named logger with a colour stream handler.

    The file handler (if set up via setup_logging) is inherited from the root
    logger automatically.

    Parameters
    ----------
    name  : Logger name — use __name__ in every module.
    level : Minimum level for this specific logger.
    """
    logger = logging.getLogger(name)
    logger.setLevel(level)

    # Avoid adding duplicate stream handlers on repeated calls
    if not any(isinstance(h, logging.StreamHandler) for h in logger.handlers):
        stream_handler = logging.StreamHandler(sys.stdout)
        stream_handler.setLevel(level)
        stream_handler.setFormatter(
            _ColourFormatter(
                "%(asctime)s | %(levelname)s | %(name)s | %(message)s",
                datefmt="%H:%M:%S",
            )
        )
        logger.addHandler(stream_handler)

    # Prevent propagation from cluttering the root with duplicate stream lines
    logger.propagate = True  # file handler on root is still captured

    return logger

def get_log_file() -> Optional[Path]:
    """Return the currently active log file path (None if not set up yet)."""
    return _log_file_path

### 3.2 Reproducibility

Seeds Python, NumPy and PyTorch (CPU + CUDA) and enables deterministic algorithms. The repository seeds everything with **42**.


In [3]:
import os
import random
import numpy as np
import torch

log = get_logger(__name__)

def seed_everything(seed: int = 42, deterministic: bool = True) -> None:
    """
    Seed all relevant RNG sources for reproducibility.

    Parameters
    ----------
    seed         : Integer seed value.
    deterministic: If True, set CUDA deterministic algorithms (may slow
                   training slightly on some ops).
    """
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # multi-GPU

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        # Available in PyTorch >= 1.8
        try:
            torch.use_deterministic_algorithms(True)
            os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        except AttributeError:
            pass  # older PyTorch — skip

    log.info(f"Global seed set to {seed} | deterministic={deterministic}")

In [4]:

# Initialise the shared run-log and set the global seed (matches repo defaults).
LOG_FILE = setup_logging(log_dir="logs", run_name="notebook_run")
seed_everything(42)
print("Logging to:", LOG_FILE)

22:31:33 | INFO     | __main__ | Global seed set to 42 | deterministic=True
Logging to: logs\notebook_run_20260605_223133.log



### 3.3 Central configuration


In [ ]:
DETECTOR_YAML_TEXT = r'''
# YOLO11s detector configuration for the COMSYS OCR baseline.
# All paths are relative to the project root unless marked absolute.


data_yaml: "data/yolo_dataset/dataset.yaml"  # generated by create_yolo_dataset.py


model: "yolo11s.pt"     # ultralytics pretrained checkpoint


imgsz: 1536
epochs: 100
batch_size: 2
optimizer: "AdamW"
lr0: 5.0e-4
lrf: 0.01              # final lr = lr0 * lrf (cosine schedule endpoint)
momentum: 0.937
weight_decay: 1.0e-4
cos_lr: true
warmup_epochs: 3
warmup_bias_lr: 0.1
freeze: 0           # freeze first N backbone layers


degrees: 10.0
translate: 0.05
scale: 0.20
perspective: 0.001
shear: 0.0

hsv_h: 0.01
hsv_s: 0.20
hsv_v: 0.20

# Disabled augmentations (would corrupt document layout)
mosaic: 0.0
mixup: 0.0
copy_paste: 0.0
fliplr: 0.0
flipud: 0.0

conf_threshold: 0.03   # very low → maximise recall
iou_threshold: 0.50    # NMS IoU


seed: 42
device: ""             # "" = auto-detect; "cpu", "0", "0,1", etc.
workers: 0
project: "outputs/detector"
run_name: "yolo11s_baseline"
save_period: 10        # save checkpoint every N epochs
patience: 50           # early-stop patience (0 = disabled)
'''


RECOGNIZER_YAML_TEXT = r'''


# ConvNeXt-Tiny recognizer configuration for the COMSYS OCR baseline.


crops_dir: "data/crops"           # created by create_crop_dataset.py
folds_dir: "data/folds"           # created by create_folds.py
fold: 0                           # which fold to train (0-4)
label_map: "data/label_map.json"  # label string → class index mapping


backbone: "convnext_small"         # timm model name
pretrained: true
input_size: 224                   # crop resize target (square)
num_classes: -1                   # -1 = auto-compute from label_map.json


epochs: 30
batch_size: 8
optimizer: "AdamW"
lr: 1.0e-4
weight_decay: 1.0e-4
label_smoothing: 0.05
use_class_weights: false           # weighted cross-entropy for imbalanced data

use_weighted_sampler: false        # WeightedRandomSampler for class balance


rotate_limit: 15
perspective_scale: 0.05           # Albumentations Perspective scale param
elastic_alpha: 1.0
elastic_sigma: 50.0
grid_distort_num_steps: 5
grid_distort_distort_limit: 0.3
brightness_contrast_limit: 0.2
gauss_noise_var_limit: [10.0, 50.0]
morph_kernel_size: 3              # kernel for dilation / erosion

# Disabled augmentations
use_mixup: false
use_cutmix: false


scheduler: "cosine"               # cosine | step | none
eta_min: 1.0e-6                   # minimum LR for cosine annealing


seed: 42
device: ""             # "" = auto-detect; "cpu", "0", etc.
workers: 0
project: "outputs/recognizer"
run_name: "convnext_small_baseline_nocw"
save_best: true
resume: ""             # path to checkpoint to resume from (empty = no resume)
'''

Path('configs').mkdir(exist_ok=True)
Path('configs/detector.yaml').write_text(DETECTOR_YAML_TEXT, encoding='utf-8')
Path('configs/recognizer.yaml').write_text(RECOGNIZER_YAML_TEXT, encoding='utf-8')

DETECTOR_CFG = yaml.safe_load(DETECTOR_YAML_TEXT)
RECOGNIZER_CFG = yaml.safe_load(RECOGNIZER_YAML_TEXT)
print('detector.yaml keys :', list(DETECTOR_CFG.keys()))
print('recognizer.yaml keys:', list(RECOGNIZER_CFG.keys()))

In [ ]:
# Detector ensemble (4 models):
DET_MAIN_WEIGHTS  = r"runs\detect\outputs\detector\yolo11m_baseline-2\weights\best.pt"
DET_FOLD0_WEIGHTS = r"runs/detect/outputs/detector/yolo11s_baseline_fold0-3/weights/best.pt"
DET_FOLD1_WEIGHTS = r"runs/detect/outputs/detector/yolo11s_baseline_fold1-2/weights/best.pt"
DET_FOLD2_WEIGHTS = r"runs/detect/outputs/detector/yolo11s_baseline_fold2-2/weights/best.pt"

# Recogniser fold ensemble (3 ConvNeXt-Small folds):
REC_FOLD0_WEIGHTS = "outputs/recognizer/convnext_small_baseline_nocw_fold0/best.pt"
REC_FOLD1_WEIGHTS = "outputs/recognizer/convnext_small_baseline_nocw_fold1/best.pt"
REC_FOLD2_WEIGHTS = "outputs/recognizer/convnext_small_baseline_nocw_fold2/best.pt"

LABEL_MAP_PATH = "data/label_map.json"

# Inference thresholds (preserved exactly from the repository defaults used in
# generate_submission / from_configs):
INFER_CONF  = 0.01     # detector confidence — very low to maximise recall
INFER_IOU   = 0.60     # per-model NMS IoU at predict time
INFER_IMGSZ = 1280     # detector inference image size
REC_INPUT   = 224      # recogniser crop input size
CROP_SIZE   = 128      # crop extraction square size before recognition
print("Checkpoint paths and inference thresholds registered.")


## 4. Dataset Analysis

The corpus is **17 annotated page images** (LabelMe JSON, one file per page)
containing **6,678 character boxes** spanning **167 unique label strings**. The
class distribution is extremely long-tailed: the most frequent class
(`U+0065`, Latin *e*) appears 633 times, while dozens of Bengali conjuncts appear
exactly once.

| Statistic | Value |
|---|---:|
| Page images | 17 |
| Annotated character boxes | 6,678 |
| Unique label strings (classes) | 167 |
| Most frequent class | `U+0065` (633) |
| Rarest classes | many singletons, e.g. `U+09C2`, `U+09AA+U+09CD+U+09B2` (1) |
| Scripts present | Latin (`U+00xx`), Bengali (`U+09xx`) |

**Top-5 classes:** `U+0065` (633), `U+0074` (452), `U+006E` (421),
`U+0069` (414), `U+0061` (405) — all common Latin letters.


### 4.1 Box utilities


In [ ]:
import numpy as np
from typing import Tuple, List


def xyxy_to_xywh(box: np.ndarray) -> np.ndarray:
    """Convert [x1, y1, x2, y2] → [x, y, w, h] (top-left + size)."""
    x1, y1, x2, y2 = box[..., 0], box[..., 1], box[..., 2], box[..., 3]
    return np.stack([x1, y1, x2 - x1, y2 - y1], axis=-1)

def xywh_to_xyxy(box: np.ndarray) -> np.ndarray:
    """Convert [x, y, w, h] → [x1, y1, x2, y2]."""
    x, y, w, h = box[..., 0], box[..., 1], box[..., 2], box[..., 3]
    return np.stack([x, y, x + w, y + h], axis=-1)

def xyxy_to_yolo(box: np.ndarray, img_w: int, img_h: int) -> np.ndarray:
    """
    Convert pixel [x1, y1, x2, y2] → normalised YOLO [cx, cy, w, h].

    Parameters
    ----------
    box   : Shape (4,) or (N, 4).
    img_w : Image width in pixels.
    img_h : Image height in pixels.
    """
    x1, y1, x2, y2 = box[..., 0], box[..., 1], box[..., 2], box[..., 3]
    cx = (x1 + x2) / 2.0 / img_w
    cy = (y1 + y2) / 2.0 / img_h
    w  = (x2 - x1) / img_w
    h  = (y2 - y1) / img_h
    return np.stack([cx, cy, w, h], axis=-1)

def yolo_to_xyxy(box: np.ndarray, img_w: int, img_h: int) -> np.ndarray:
    """
    Convert normalised YOLO [cx, cy, w, h] → pixel [x1, y1, x2, y2].

    Parameters
    ----------
    box   : Shape (4,) or (N, 4).
    img_w : Image width in pixels.
    img_h : Image height in pixels.
    """
    cx, cy, w, h = box[..., 0], box[..., 1], box[..., 2], box[..., 3]
    x1 = (cx - w / 2) * img_w
    y1 = (cy - h / 2) * img_h
    x2 = (cx + w / 2) * img_w
    y2 = (cy + h / 2) * img_h
    return np.stack([x1, y1, x2, y2], axis=-1)

def points_to_xyxy(points: List[List[float]]) -> np.ndarray:
    """
    Convert LabelMe 'points' [[x1,y1],[x2,y2]] → [x1, y1, x2, y2].

    Handles both orderings (top-left / bottom-right) by taking min/max.
    """
    pts = np.array(points, dtype=np.float32)
    x1, y1 = pts[:, 0].min(), pts[:, 1].min()
    x2, y2 = pts[:, 0].max(), pts[:, 1].max()
    return np.array([x1, y1, x2, y2], dtype=np.float32)


def iou_matrix(boxes_a: np.ndarray, boxes_b: np.ndarray) -> np.ndarray:
    """
    Compute pairwise IoU between two sets of boxes.

    Parameters
    ----------
    boxes_a : Shape (M, 4) in xyxy format.
    boxes_b : Shape (N, 4) in xyxy format.

    Returns
    -------
    iou : Shape (M, N) float32 array.
    """
    # Expand dims for broadcasting: (M, 1, 4) vs (1, N, 4)
    a = boxes_a[:, None, :]   # (M, 1, 4)
    b = boxes_b[None, :, :]   # (1, N, 4)

    inter_x1 = np.maximum(a[..., 0], b[..., 0])
    inter_y1 = np.maximum(a[..., 1], b[..., 1])
    inter_x2 = np.minimum(a[..., 2], b[..., 2])
    inter_y2 = np.minimum(a[..., 3], b[..., 3])

    inter_w = np.maximum(0.0, inter_x2 - inter_x1)
    inter_h = np.maximum(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = (boxes_a[:, 2] - boxes_a[:, 0]) * (boxes_a[:, 3] - boxes_a[:, 1])
    area_b = (boxes_b[:, 2] - boxes_b[:, 0]) * (boxes_b[:, 3] - boxes_b[:, 1])

    union_area = area_a[:, None] + area_b[None, :] - inter_area
    union_area = np.maximum(union_area, 1e-6)  # avoid div-by-zero

    return (inter_area / union_area).astype(np.float32)

def iou_single(box_a: np.ndarray, box_b: np.ndarray) -> float:
    """Scalar IoU between two single boxes (xyxy format)."""
    return float(iou_matrix(box_a[None], box_b[None])[0, 0])


def crop_with_padding(
    image: np.ndarray,
    box: np.ndarray,
    target_size: int = 128,
    pad_value: int = 255,
) -> np.ndarray:
    """
    Extract a bounding-box crop from an image with aspect-ratio-preserving
    padding, then resize to target_size × target_size.

    Parameters
    ----------
    image       : HxWxC uint8 NumPy array (BGR or RGB, doesn't matter here).
    box         : [x1, y1, x2, y2] in pixels.
    target_size : Output square side length.
    pad_value   : Constant pad fill value (255 = white background).

    Returns
    -------
    crop : uint8 NumPy array of shape (target_size, target_size, C).
    """
    import cv2

    H, W = image.shape[:2]
    x1, y1, x2, y2 = (
        max(0, int(round(box[0]))),
        max(0, int(round(box[1]))),
        min(W, int(round(box[2]))),
        min(H, int(round(box[3]))),
    )

    # Guard against degenerate boxes
    if x2 <= x1 or y2 <= y1:
        return np.full(
            (target_size, target_size, image.shape[2] if image.ndim == 3 else 1),
            pad_value,
            dtype=np.uint8,
        )

    margin = 0.08

    bw = x2 - x1
    bh = y2 - y1

    x1 = max(0, int(x1 - bw * margin))
    y1 = max(0, int(y1 - bh * margin))
    x2 = min(W, int(x2 + bw * margin))
    y2 = min(H, int(y2 + bh * margin))
    
    crop = image[y1:y2, x1:x2]
    ch, cw = crop.shape[:2]

    # Pad to square preserving aspect ratio
    max_dim = max(ch, cw)
    pad_h = max_dim - ch
    pad_w = max_dim - cw
    top    = pad_h // 2
    bottom = pad_h - top
    left   = pad_w // 2
    right  = pad_w - left

    crop_sq = cv2.copyMakeBorder(
        crop, top, bottom, left, right,
        cv2.BORDER_CONSTANT, value=pad_value
    )

    crop_resized = cv2.resize(
        crop_sq, (target_size, target_size), interpolation=cv2.INTER_LINEAR
    )
    return crop_resized

def clip_boxes(boxes: np.ndarray, img_w: int, img_h: int) -> np.ndarray:
    """Clip xyxy boxes to image boundaries."""
    boxes = boxes.copy()
    boxes[:, 0] = np.clip(boxes[:, 0], 0, img_w)
    boxes[:, 1] = np.clip(boxes[:, 1], 0, img_h)
    boxes[:, 2] = np.clip(boxes[:, 2], 0, img_w)
    boxes[:, 3] = np.clip(boxes[:, 3], 0, img_h)
    return boxes

### 4.2 Visualisation helpers

Used for debugging overlays and class-distribution charts.


In [ ]:
import math
import cv2
import numpy as np
from typing import List, Optional, Tuple, Dict

_PALETTE: List[Tuple[int, int, int]] = [
    (0,   255,   0),   # green  – ground truth / default
    (0,   128, 255),   # orange – predictions
    (255,   0,   0),   # blue
    (0,   255, 255),   # yellow
    (255,   0, 255),   # magenta
]

def draw_boxes(
    image: np.ndarray,
    boxes: np.ndarray,
    labels: Optional[List[str]] = None,
    color: Tuple[int, int, int] = (0, 255, 0),
    thickness: int = 2,
    font_scale: float = 0.5,
) -> np.ndarray:
    """
    Draw bounding boxes (xyxy pixel coords) onto a copy of the image.

    Parameters
    ----------
    image      : HxWxC uint8.
    boxes      : (N, 4) float array, xyxy pixels.
    labels     : Optional list of N label strings.
    color      : BGR colour tuple.
    thickness  : Line thickness.
    font_scale : OpenCV font scale for labels.

    Returns
    -------
    Annotated image copy (uint8).
    """
    out = image.copy()
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])
        cv2.rectangle(out, (x1, y1), (x2, y2), color, thickness)
        if labels is not None and i < len(labels):
            txt = str(labels[i])
            (tw, th), _ = cv2.getTextSize(
                txt, cv2.FONT_HERSHEY_SIMPLEX, font_scale, 1
            )
            cv2.rectangle(out, (x1, y1 - th - 4), (x1 + tw + 2, y1), color, -1)
            cv2.putText(
                out, txt, (x1 + 1, y1 - 2),
                cv2.FONT_HERSHEY_SIMPLEX, font_scale,
                (0, 0, 0), 1, cv2.LINE_AA,
            )
    return out

def draw_boxes_dual(
    image: np.ndarray,
    gt_boxes: np.ndarray,
    pred_boxes: np.ndarray,
    pred_labels: Optional[List[str]] = None,
) -> np.ndarray:
    """
    Draw ground-truth boxes in green and predicted boxes in orange.

    Parameters
    ----------
    image       : HxWxC uint8.
    gt_boxes    : (M, 4) ground-truth boxes in xyxy pixels.
    pred_boxes  : (N, 4) predicted boxes in xyxy pixels.
    pred_labels : Optional list of N prediction label strings.

    Returns
    -------
    Annotated image copy.
    """
    out = draw_boxes(image, gt_boxes, color=_PALETTE[0], thickness=2)
    out = draw_boxes(out, pred_boxes, labels=pred_labels,
                     color=_PALETTE[1], thickness=1)
    return out

def visualize_crops(
    crops: List[np.ndarray],
    labels: List[str],
    grid_cols: int = 10,
    cell_size: int = 80,
    bg_color: int = 240,
) -> np.ndarray:
    """
    Arrange a list of crop images into a grid with label text below each.

    Parameters
    ----------
    crops      : List of HxWxC uint8 images (will be resized to cell_size).
    labels     : Corresponding label strings.
    grid_cols  : Number of columns in the grid.
    cell_size  : Side length of each cell in pixels.
    bg_color   : Background grey level.

    Returns
    -------
    Grid image (uint8 RGB).
    """
    n = len(crops)
    if n == 0:
        return np.full((cell_size, cell_size, 3), bg_color, dtype=np.uint8)

    grid_rows = math.ceil(n / grid_cols)
    label_height = 18
    total_h = grid_rows * (cell_size + label_height)
    total_w = grid_cols * cell_size
    canvas = np.full((total_h, total_w, 3), bg_color, dtype=np.uint8)

    for idx, (crop, lbl) in enumerate(zip(crops, labels)):
        row = idx // grid_cols
        col = idx % grid_cols
        y_off = row * (cell_size + label_height)
        x_off = col * cell_size

        # Resize crop
        if crop.ndim == 2:
            crop = cv2.cvtColor(crop, cv2.COLOR_GRAY2BGR)
        cell = cv2.resize(crop, (cell_size, cell_size))
        canvas[y_off: y_off + cell_size, x_off: x_off + cell_size] = cell

        # Draw label
        txt = lbl[:12]  # truncate long labels
        cv2.putText(
            canvas, txt,
            (x_off + 2, y_off + cell_size + label_height - 3),
            cv2.FONT_HERSHEY_SIMPLEX, 0.3, (50, 50, 50), 1, cv2.LINE_AA,
        )

    return canvas

def plot_class_distribution(
    label_counts: Dict[str, int],
    top_k: int = 50,
    bar_width: int = 20,
    bar_max_height: int = 300,
    margin: int = 40,
) -> np.ndarray:
    """
    Simple bar chart of class frequency (top-k classes).

    Parameters
    ----------
    label_counts   : Dict mapping label → count.
    top_k          : Show only the top_k most frequent classes.
    bar_width      : Width of each bar in pixels.
    bar_max_height : Maximum bar height in pixels.
    margin         : Left / bottom margin in pixels.

    Returns
    -------
    Bar chart image (uint8 BGR).
    """
    sorted_items = sorted(label_counts.items(), key=lambda x: x[1], reverse=True)
    top = sorted_items[:top_k]
    if not top:
        return np.full((100, 100, 3), 255, dtype=np.uint8)

    max_count = top[0][1]
    n = len(top)
    w = margin + n * bar_width + margin
    h = margin + bar_max_height + 30
    canvas = np.full((h, w, 3), 255, dtype=np.uint8)

    for i, (label, count) in enumerate(top):
        bar_h = int(count / max_count * bar_max_height) if max_count > 0 else 0
        x1 = margin + i * bar_width
        x2 = x1 + bar_width - 2
        y1 = margin + bar_max_height - bar_h
        y2 = margin + bar_max_height
        cv2.rectangle(canvas, (x1, y1), (x2, y2), (70, 130, 220), -1)

    # Axes
    cv2.line(canvas,
             (margin, margin), (margin, margin + bar_max_height), (0, 0, 0), 1)
    cv2.line(canvas,
             (margin, margin + bar_max_height),
             (w - margin, margin + bar_max_height), (0, 0, 0), 1)

    # Title
    cv2.putText(canvas, f"Top-{top_k} class distribution",
                (margin, margin - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

    return canvas

### 4.3 LabelMe parsing

Parses all LabelMe JSON files into one unified DataFrame (one row per box), sorted by `image_file, y1, x1` (reading order). This is the source of truth for both the detector and recogniser datasets.


In [ ]:
"""
preprocessing/convert_labelme.py
---------------------------------
Parse all LabelMe JSON annotation files in a directory and emit a unified
pandas DataFrame with one row per annotated character box.

Schema of output DataFrame / CSV
---------------------------------
    image_file  : basename of the source page image (e.g. "page_01.jpg")
    image_path  : absolute path to the page image
    label       : raw label string from LabelMe (e.g. "U+0065" or
                  "U+0069+U+0069")
    x1, y1     : top-left corner of bounding box (pixels)
    x2, y2     : bottom-right corner of bounding box (pixels)
    width       : box width  (x2 - x1)
    height      : box height (y2 - y1)

Usage (CLI)
-----------
    python -m src.preprocessing.convert_labelme \\
        --ann_dir  data/raw/annotations \\
        --img_dir  data/raw/images \\
        --out_csv  data/annotations.csv

Usage (Python API)
------------------
    df = parse_labelme_dir("data/raw/annotations", "data/raw/images")
"""

import argparse
import json
import sys
from pathlib import Path
from typing import List, Optional

import pandas as pd

# Local imports — adjust sys.path when running as __main__

log = get_logger(__name__)

_IMG_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

def _find_image(img_dir: Path, stem: str) -> Optional[Path]:
    """
    Locate a page image by its stem (filename without extension).
    Returns the first match across common image extensions.
    """
    for ext in _IMG_EXTS:
        p = img_dir / (stem + ext)
        if p.exists():
            return p
    return None

def parse_labelme_json(
    json_path: Path,
    img_dir: Path,
) -> List[dict]:
    """
    Parse a single LabelMe JSON file.

    Parameters
    ----------
    json_path : Path to the *.json annotation file.
    img_dir   : Directory containing the corresponding page images.

    Returns
    -------
    List of dicts, one per annotated shape (character box).
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # LabelMe stores the image filename in the 'imagePath' key
    raw_img_name = Path(data.get("imagePath", "")).name
    stem = Path(raw_img_name).stem

    img_path = _find_image(img_dir, stem)
    if img_path is None:
        # Fall back: try the same directory as the JSON
        img_path = _find_image(json_path.parent, stem)
    if img_path is None:
        log.warning(
            f"[convert_labelme] Image not found for JSON '{json_path.name}' "
            f"(stem='{stem}'). Rows will have null image_path."
        )

    rows: List[dict] = []
    shapes = data.get("shapes", [])

    for shape in shapes:
        if shape.get("shape_type") != "rectangle":
            # Skip non-rectangle annotations (e.g. polygons) if present
            continue

        label: str = shape.get("label", "").strip()
        points = shape.get("points", [])

        if len(points) < 2:
            log.warning(
                f"[convert_labelme] Skipping shape with < 2 points in {json_path.name}"
            )
            continue

        try:
            box = points_to_xyxy(points)
        except Exception as exc:
            log.warning(
                f"[convert_labelme] Failed to parse points in {json_path.name}: {exc}"
            )
            continue

        x1, y1, x2, y2 = float(box[0]), float(box[1]), float(box[2]), float(box[3])

        rows.append(
            {
                "image_file": raw_img_name,
                "image_path": str(img_path) if img_path else None,
                "json_file":  json_path.name,
                "label":      label,
                "x1":         x1,
                "y1":         y1,
                "x2":         x2,
                "y2":         y2,
                "width":      x2 - x1,
                "height":     y2 - y1,
            }
        )

    return rows

def parse_labelme_dir(
    ann_dir: str | Path,
    img_dir: str | Path,
) -> pd.DataFrame:
    """
    Parse all LabelMe JSON files in *ann_dir*.

    Parameters
    ----------
    ann_dir : Directory of LabelMe JSON files.
    img_dir : Directory of corresponding page images.

    Returns
    -------
    Unified DataFrame sorted by image_file then y1 (reading order).
    """
    ann_dir = Path(ann_dir)
    img_dir = Path(img_dir)

    json_files = sorted(ann_dir.glob("*.json"))
    if not json_files:
        log.error(f"No JSON files found in '{ann_dir}'")
        return pd.DataFrame()

    log.info(f"Found {len(json_files)} JSON annotation file(s) in '{ann_dir}'")

    all_rows: List[dict] = []
    for jf in json_files:
        rows = parse_labelme_json(jf, img_dir)
        log.info(f"  {jf.name}: {len(rows)} boxes parsed")
        all_rows.extend(rows)

    if not all_rows:
        log.error("No valid annotation rows parsed — check annotation format.")
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)
    df = df.sort_values(["image_file", "y1", "x1"]).reset_index(drop=True)

    n_images = df["image_file"].nunique()
    n_labels = df["label"].nunique()
    log.info(
        f"Parsed {len(df)} total boxes | "
        f"{n_images} images | "
        f"{n_labels} unique label strings"
    )

    return df

In [ ]:

# Mirrors:  python -m src.preprocessing.convert_labelme \
#               --ann_dir data/raw/annotations --img_dir data/raw/images \
#               --out_csv data/annotations.csv
RAW_ANN_DIR = "data/raw/annotations"
RAW_IMG_DIR = "data/raw/images"
ANNOTATIONS_CSV = "data/annotations.csv"

if Path(RAW_ANN_DIR).exists() and Path(RAW_IMG_DIR).exists():
    ann_df = parse_labelme_dir(RAW_ANN_DIR, RAW_IMG_DIR)
    Path("data").mkdir(parents=True, exist_ok=True)
    ann_df.to_csv(ANNOTATIONS_CSV, index=False)

    # ── Exploratory summary ────────────────────────────────────────────────────
    print("Pages              :", ann_df["image_file"].nunique())
    print("Total boxes        :", len(ann_df))
    print("Unique label strings:", ann_df["label"].nunique())
    freq = ann_df["label"].value_counts()
    print("\nTop-10 classes:\n", freq.head(10).to_string())
    print("\nNumber of singleton classes:", int((freq == 1).sum()))
else:
    print(f"Raw data not found at '{RAW_ANN_DIR}' / '{RAW_IMG_DIR}'.")
    print("Skipping live EDA — see the dataset statistics table above.")


## 5. Detection Pipeline

The detector is a **single-class** YOLO11 model — every character is class 0
(`"character"`); the actual identity is left to Stage 2. This keeps the detection
problem data-efficient given only 17 pages.

**Augmentation policy (document-safe).** Layout-destroying augmentations are
disabled (`mosaic=0`, `mixup=0`, `copy_paste=0`, `fliplr=0`, `flipud=0`) because
flipping or mosaicking a page of text produces nonsense. Only mild affine and
HSV jitter are kept (`degrees=10`, `translate=0.05`, `scale=0.20`,
`perspective=0.001`).


### 5.1 YOLO dataset preparation

Converts the annotations CSV into the on-disk YOLO layout (images/labels split, normalised `cx cy w h` label files, `dataset.yaml`). Image-level train/val split with seed 42.

In [ ]:
"""
preprocessing/create_yolo_dataset.py
--------------------------------------
Convert the unified annotations CSV into a YOLO-format dataset on disk.

Output layout
-------------
    data/yolo_dataset/
        images/
            train/   <- symlinks or copies of page images
            val/
        labels/
            train/   <- one .txt per image (YOLO normalised xywh)
            val/
        dataset.yaml <- dataset descriptor for ultralytics

All characters share class 0 ("character").  Recognition is handled by the
separate ConvNeXt classifier.

Usage (CLI)
-----------
    python -m src.preprocessing.create_yolo_dataset \\
        --ann_csv   data/annotations.csv \\
        --out_dir   data/yolo_dataset \\
        --val_split 0.15

Usage (Python API)
------------------
    create_yolo_dataset("data/annotations.csv", "data/yolo_dataset")
"""

import argparse
import shutil
import sys
from pathlib import Path
from typing import Optional, List

import cv2
import pandas as pd
import yaml

log = get_logger(__name__)

CLASS_ID = 0
CLASS_NAME = "character"

def _write_yolo_label(
    label_path: Path,
    boxes_xyxy: pd.DataFrame,
    img_w: int,
    img_h: int,
) -> None:
    """Write one YOLO .txt label file for a single image."""
    lines: List[str] = []
    for _, row in boxes_xyxy.iterrows():
        box = xyxy_to_yolo(
            __import__("numpy").array([row["x1"], row["y1"], row["x2"], row["y2"]]),
            img_w,
            img_h,
        )
        cx, cy, w, h = box
        # Clamp to [0, 1] — small violations can occur due to annotation error
        cx, cy, w, h = (
            max(0.0, min(1.0, float(cx))),
            max(0.0, min(1.0, float(cy))),
            max(1e-4, min(1.0, float(w))),
            max(1e-4, min(1.0, float(h))),
        )
        lines.append(f"{CLASS_ID} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    label_path.write_text("\n".join(lines), encoding="utf-8")

def create_yolo_dataset(
    ann_csv: str | Path,
    out_dir: str | Path,
    val_split: float = 0.15,
    copy_images: bool = True,
    seed: int = 42,
) -> Path:
    """
    Convert annotations CSV → YOLO-format dataset directory.

    Parameters
    ----------
    ann_csv    : Path to the unified annotations CSV.
    out_dir    : Root output directory.
    val_split  : Fraction of *images* reserved for validation.
    copy_images: If True, copy images; else create relative symlinks.
    seed       : Random seed for train/val split.

    Returns
    -------
    Path to the generated dataset.yaml.
    """
    import numpy as np

    ann_csv = Path(ann_csv)
    out_dir = Path(out_dir)

    df = pd.read_csv(ann_csv)
    log.info(f"Loaded {len(df)} annotation rows from '{ann_csv}'")


    images = df["image_file"].unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(images)
    n_val = max(1, int(len(images) * val_split))
    val_images = set(images[:n_val])
    train_images = set(images[n_val:])
    log.info(
        f"Split: {len(train_images)} train images / {len(val_images)} val images"
    )


    for split in ("train", "val"):
        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)


    total_boxes = {"train": 0, "val": 0}

    for img_file in sorted(df["image_file"].unique()):
        split = "val" if img_file in val_images else "train"
        img_rows = df[df["image_file"] == img_file]

        # Locate source image (use first non-null image_path in the group)
        src_paths = img_rows["image_path"].dropna().unique()
        if len(src_paths) == 0:
            log.warning(f"  No image_path found for '{img_file}' — skipping")
            continue
        src_img = Path(src_paths[0])
        if not src_img.exists():
            log.warning(f"  Image not found: '{src_img}' — skipping")
            continue

        # Read image dimensions (needed for YOLO normalisation)
        img_cv = cv2.imread(str(src_img))
        if img_cv is None:
            log.warning(f"  cv2.imread failed for '{src_img}' — skipping")
            continue
        img_h, img_w = img_cv.shape[:2]

        # Copy / symlink image
        dst_img = out_dir / "images" / split / src_img.name
        if copy_images:
            shutil.copy2(src_img, dst_img)
        else:
            if not dst_img.exists():
                dst_img.symlink_to(src_img.resolve())

        # Write label file
        label_path = (
            out_dir / "labels" / split / (src_img.stem + ".txt")
        )
        _write_yolo_label(label_path, img_rows, img_w, img_h)
        total_boxes[split] += len(img_rows)

        log.debug(f"  [{split}] {img_file}: {len(img_rows)} boxes written")

    log.info(
        f"Boxes written — train: {total_boxes['train']}, val: {total_boxes['val']}"
    )

    dataset_yaml = {
        "path": str(out_dir.resolve()),
        "train": "images/train",
        "val":   "images/val",
        "nc":    1,
        "names": [CLASS_NAME],
    }
    yaml_path = out_dir / "dataset.yaml"
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(dataset_yaml, f, default_flow_style=False, sort_keys=False)
    log.info(f"Wrote dataset descriptor → '{yaml_path}'")

    return yaml_path

### 5.2 Detector cross-validation folds

Image-level **3-fold** split (`KFold`, seed 42) producing `dataset_fold{0,1,2}.yaml`. These are the folds whose checkpoints feed the detector ensemble. (Run after `create_yolo_dataset`.)


In [ ]:
from pathlib import Path
from sklearn.model_selection import KFold
import shutil
import yaml

ROOT = Path("data/yolo_dataset")

N_FOLDS = 3
SEED = 42

def main():

    train_imgs = list((ROOT / "images/train").glob("*"))
    val_imgs   = list((ROOT / "images/val").glob("*"))

    all_imgs = sorted(train_imgs + val_imgs)

    stems = [p.stem for p in all_imgs]

    kf = KFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=SEED,
    )

    for fold, (train_idx, val_idx) in enumerate(kf.split(stems)):

        fold_root = ROOT / f"fold{fold}"

        img_train = fold_root / "images/train"
        img_val   = fold_root / "images/val"

        lbl_train = fold_root / "labels/train"
        lbl_val   = fold_root / "labels/val"

        for d in [
            img_train, img_val,
            lbl_train, lbl_val,
        ]:
            d.mkdir(parents=True, exist_ok=True)

        train_stems = [stems[i] for i in train_idx]
        val_stems   = [stems[i] for i in val_idx]

        for stem in train_stems:

            img = next(p for p in all_imgs if p.stem == stem)

            lbl = (
                ROOT / "labels/train" / f"{stem}.txt"
            )

            if not lbl.exists():
                lbl = ROOT / "labels/val" / f"{stem}.txt"

            shutil.copy2(img, img_train / img.name)
            shutil.copy2(lbl, lbl_train / lbl.name)

        for stem in val_stems:

            img = next(p for p in all_imgs if p.stem == stem)

            lbl = (
                ROOT / "labels/train" / f"{stem}.txt"
            )

            if not lbl.exists():
                lbl = ROOT / "labels/val" / f"{stem}.txt"

            shutil.copy2(img, img_val / img.name)
            shutil.copy2(lbl, lbl_val / lbl.name)

        yaml_path = ROOT / f"dataset_fold{fold}.yaml"

        yaml_data = {
            "path": str(fold_root.resolve()),
            "train": "images/train",
            "val": "images/val",
            "names": {
                0: "character"
            }
        }

        with open(yaml_path, "w") as f:
            yaml.safe_dump(yaml_data, f)

        print(
            f"Fold {fold}: "
            f"{len(train_stems)} train | "
            f"{len(val_stems)} val"
        )

### 5.3 Detector training

Reads `configs/detector.yaml`, optionally selects a fold-specific `dataset_fold{k}.yaml`, and trains via Ultralytics. All hyperparameters (lr0=5e-4, AdamW, cosine LR, 100 epochs, imgsz=1536, batch=2) come straight from the YAML.


In [ ]:
"""
detector/train_detector.py
---------------------------
Train a YOLO11s character detector using Ultralytics.

This script:
1. Reads configs/detector.yaml
2. Optionally resumes from a checkpoint
3. Runs cross-validation at image level (one fold per run)
4. Logs mAP, precision, recall after every epoch
5. Saves the best checkpoint to outputs/detector/<run_name>/

Usage (CLI)
-----------
    python -m src.detector.train_detector \\
        --config configs/detector.yaml \\
        --fold   0           # 0-4 for cross-validation, -1 = full dataset

    # Resume:
    python -m src.detector.train_detector \\
        --config configs/detector.yaml \\
        --resume outputs/detector/yolo11s_baseline/fold0/weights/last.pt
"""

import argparse
import sys
from pathlib import Path
from typing import Optional, List

import yaml

log = get_logger(__name__)

def load_config(config_path: str | Path) -> dict:
    with open(config_path, "r") as f:
        return yaml.safe_load(f)

def train_detector(
    config_path: str | Path,
    fold: int = -1,
    resume: Optional[str] = None,
) -> None:
    """
    Train YOLO11s on the prepared dataset.

    Parameters
    ----------
    config_path : Path to configs/detector.yaml.
    fold        : Fold index for cross-validation.  -1 means use the full
                  dataset split defined in dataset.yaml.
    resume      : Optional path to a YOLO checkpoint to resume from.
    """
    try:
        from ultralytics import YOLO
    except ImportError:
        log.error("Ultralytics not installed.  Run: pip install ultralytics")
        sys.exit(1)

    cfg = load_config(config_path)
    seed_everything(cfg.get("seed", 42))


    data_yaml = cfg["data_yaml"]

    # If running cross-validation, a fold-specific dataset YAML should exist.
    # create_yolo_dataset.py can be extended to write per-fold YAMLs; for now
    # we use the single dataset.yaml and accept image-level random split.
    if fold >= 0:
        fold_yaml = Path(data_yaml).parent / f"dataset_fold{fold}.yaml"
        if fold_yaml.exists():
            data_yaml = str(fold_yaml)
            log.info(f"Using fold-specific dataset YAML: '{fold_yaml}'")
        else:
            log.warning(
                f"Fold-specific YAML not found ('{fold_yaml}'). "
                f"Falling back to '{data_yaml}'"
            )


    run_name = cfg.get("run_name", "yolo11s_baseline")
    if fold >= 0:
        run_name = f"{run_name}_fold{fold}"

    project = cfg.get("project", "outputs/detector")


    model_path = cfg.get("model", "yolo11s.pt")

    if resume:
        log.info(f"Resuming from checkpoint: '{resume}'")
        model = YOLO(resume)
    else:
        log.info(f"Initialising model from: '{model_path}'")
        model = YOLO(model_path)


    train_args = {
        # Data
        "data":          data_yaml,
        # Architecture
        "imgsz":         cfg.get("imgsz", 1536),
        # Optimisation
        "epochs":        cfg.get("epochs", 200),
        "batch":         cfg.get("batch_size", 8),
        "optimizer":     cfg.get("optimizer", "AdamW"),
        "lr0":           cfg.get("lr0", 5e-4),
        "lrf":           cfg.get("lrf", 0.01),
        "momentum":      cfg.get("momentum", 0.937),
        "weight_decay":  cfg.get("weight_decay", 1e-4),
        "cos_lr":        cfg.get("cos_lr", True),
        "warmup_epochs": cfg.get("warmup_epochs", 3),
        "warmup_bias_lr":cfg.get("warmup_bias_lr", 0.1),
        "freeze":        cfg.get("freeze", 0),
        # Augmentation
        "degrees":       cfg.get("degrees", 10.0),
        "translate":     cfg.get("translate", 0.05),
        "scale":         cfg.get("scale", 0.20),
        "perspective":   cfg.get("perspective", 0.001),
        "shear":         cfg.get("shear", 0.0),
        "hsv_h":         cfg.get("hsv_h", 0.01),
        "hsv_s":         cfg.get("hsv_s", 0.20),
        "hsv_v":         cfg.get("hsv_v", 0.20),
        "mosaic":        cfg.get("mosaic", 0.0),
        "mixup":         cfg.get("mixup", 0.0),
        "copy_paste":    cfg.get("copy_paste", 0.0),
        "fliplr":        cfg.get("fliplr", 0.0),
        "flipud":        cfg.get("flipud", 0.0),
        # Misc
        "device":        cfg.get("device", ""),
        "workers":       cfg.get("workers", 4),
        "seed":          cfg.get("seed", 42),
        "project":       project,
        "name":          run_name,
        "save_period":   cfg.get("save_period", 10),
        "patience":      cfg.get("patience", 40),
        "exist_ok":      bool(resume),  # allow overwriting dir when resuming
        "resume":        bool(resume),
        "plots":         True,
        "verbose":       True,
    }

    log.info("=== Starting YOLO11s training ===")
    log.info(f"Project: '{project}' | Run: '{run_name}'")
    log.info(f"Epochs: {train_args['epochs']} | imgsz: {train_args['imgsz']}")
    log.info(f"Batch:  {train_args['batch']}  | Device: {train_args['device'] or 'auto'}")

    results = model.train(**train_args)

    try:
        box   = results.results_dict
        map50 = box.get("metrics/mAP50(B)", "N/A")
        map50_95 = box.get("metrics/mAP50-95(B)", "N/A")
        p     = box.get("metrics/precision(B)", "N/A")
        r     = box.get("metrics/recall(B)", "N/A")
        log.info(
            f"=== Training Complete ===\n"
            f"  mAP@50:      {map50}\n"
            f"  mAP@50-95:   {map50_95}\n"
            f"  Precision:   {p}\n"
            f"  Recall:      {r}"
        )
    except Exception:
        log.info("=== Training Complete (metrics unavailable) ===")

In [ ]:

# ── Train one detector fold (uncomment to run; needs the YOLO dataset on disk) ──
# Mirrors:  python -m src.detector.train_detector --config configs/detector.yaml --fold 0
#
# train_detector(config_path="configs/detector.yaml", fold=0)
# train_detector(config_path="configs/detector.yaml", fold=1)
# train_detector(config_path="configs/detector.yaml", fold=2)
print("Detector training entry point ready (call train_detector(...) to launch).")


### 5.4 Detector inference (production ensemble)

This is the **active** inference class. On construction it loads four YOLO models
(the main checkpoint passed in, plus three fixed fold checkpoints). `_predict_raw`
runs all four with test-time augmentation (`augment=True`, `max_det=6000`),
concatenates every box, and de-duplicates with `torchvision.ops.nms` at **IoU
0.85**. The high NMS threshold is deliberate: it only removes boxes that almost
perfectly coincide, so complementary detections from different models are kept,
maximising recall.

> The fold checkpoint paths are the exact strings from the repository. They are
> the only file dependency of this class beyond the main weights argument.


In [ ]:
"""
detector/infer_detector.py
---------------------------
Run YOLO11s inference on one or more page images and return predicted boxes.

Outputs
-------
For each image the function returns:
    List[DetectionResult] where each DetectionResult has:
        image_path : str
        boxes      : np.ndarray  (N, 4) xyxy pixels
        scores     : np.ndarray  (N,)   confidence scores
        labels     : List[str]   always ["character"] * N (detector is single-class)

Usage (CLI)
-----------
    python -m src.detector.infer_detector \\
        --weights  outputs/detector/yolo11s_baseline/weights/best.pt \\
        --images   data/raw/images \\
        --out_dir  outputs/detector_preds \\
        --conf     0.03 \\
        --iou      0.50

Usage (Python API)
------------------
    detector = DetectorInference("path/to/best.pt", conf=0.03, iou=0.50)
    results = detector.predict_image("page.png")
"""

import argparse
import json
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional, Union
import torch
import torchvision

import cv2
import numpy as np
from tqdm import tqdm

log = get_logger(__name__)

_IMG_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

@dataclass
class DetectionResult:
    """Container for YOLO inference results on a single image."""
    image_path: str
    image_w: int
    image_h: int
    boxes: np.ndarray   # (N, 4) xyxy float32 pixels
    scores: np.ndarray  # (N,) float32 confidence
    class_ids: np.ndarray = field(default_factory=lambda: np.array([]))

    def __post_init__(self) -> None:
        if self.class_ids.size == 0 and self.boxes.size > 0:
            self.class_ids = np.zeros(len(self.boxes), dtype=np.int32)

    @property
    def num_boxes(self) -> int:
        return len(self.boxes)

    def to_dict(self) -> dict:
        return {
            "image_path": self.image_path,
            "image_w":    self.image_w,
            "image_h":    self.image_h,
            "boxes":      self.boxes.tolist(),
            "scores":     self.scores.tolist(),
            "class_ids":  self.class_ids.tolist(),
        }

class DetectorInference:
    """
    YOLO-based character detector wrapper.

    Parameters
    ----------
    weights_path : Path to trained YOLO weights (.pt).
    conf         : Confidence threshold.  Low values → high recall.
    iou          : NMS IoU threshold.
    imgsz        : Inference image size.
    device       : Torch device string ("", "cpu", "0", etc.).
    """

    def __init__(
        self,
        weights_path: str | Path,
        conf: float = 0.03,
        iou: float = 0.50,
        imgsz: int = 1280,
        device: str = "",
    ) -> None:
        try:
            from ultralytics import YOLO
        except ImportError:
            log.error("Ultralytics not installed.  pip install ultralytics")
            sys.exit(1)

        self.weights_path = Path(weights_path)
        self.conf  = conf
        self.iou   = iou
        self.imgsz = imgsz
        self.device = device

        log.info(f"Loading detector weights: '{self.weights_path}'")
        self._model_m = YOLO(str(self.weights_path))

        self._model_f0 = YOLO(
            r"runs/detect/outputs/detector/yolo11s_baseline_fold0-3/weights/best.pt"
        )

        self._model_f1 = YOLO(
            r"runs/detect/outputs/detector/yolo11s_baseline_fold1-2/weights/best.pt"
        )

        self._model_f2 = YOLO(
            r"runs/detect/outputs/detector/yolo11s_baseline_fold2-2/weights/best.pt"
        )
        log.info(
            f"Detector ready | conf={conf} | iou={iou} | imgsz={imgsz}"
        )

    
    def _extract(self,result):
        if result.boxes is None or len(result.boxes) == 0:
            return (
                np.empty((0,4), dtype=np.float32),
                np.empty(0, dtype=np.float32),
                np.empty(0, dtype=np.int32)
            )

        return (
            result.boxes.xyxy.cpu().numpy().astype(np.float32),
            result.boxes.conf.cpu().numpy().astype(np.float32),
            result.boxes.cls.cpu().numpy().astype(np.int32),
        )

    def _predict_raw(self, image):

        results_f0 = self._model_f0.predict(
            source=image,
            conf=self.conf,
            iou=self.iou,
            imgsz=self.imgsz,
            device=self.device,
            max_det=6000,
            verbose=False,
            augment=True,
        )
        
        results_f1 = self._model_f1.predict(
            source=image,
            conf=self.conf,
            iou=self.iou,
            imgsz=self.imgsz,
            device=self.device,
            max_det=6000,
            verbose=False,
            augment=True,
        )
        
        results_f2 = self._model_f2.predict(
            source=image,
            conf=self.conf,
            iou=self.iou,
            imgsz=self.imgsz,
            device=self.device,
            max_det=6000,
            verbose=False,
            augment=True,
        )

        results_m = self._model_m.predict(
            source=image,
            conf=self.conf,
            iou=self.iou,
            imgsz=self.imgsz,
            device=self.device,
            max_det=6000,
            verbose=False,
            augment=True,
        )

        r_f0 = results_f0[0]
        r_f1 = results_f1[0]
        r_f2 = results_f2[0]
        r_m  = results_m[0]

        boxes_f0, scores_f0, cls_f0 = self._extract(r_f0)
        boxes_f1, scores_f1, cls_f1 = self._extract(r_f1)
        boxes_f2, scores_f2, cls_f2 = self._extract(r_f2)
        boxes_m , scores_m , cls_m  = self._extract(r_m)

        boxes = np.concatenate([
            boxes_f0,
            boxes_f1,
            boxes_f2,
            boxes_m
        ], axis=0)

        scores = np.concatenate([
            scores_f0,
            scores_f1,
            scores_f2,
            scores_m
        ], axis=0)

        cls = np.concatenate([
            cls_f0,
            cls_f1,
            cls_f2,
            cls_m
        ], axis=0)

        print(
            f"[FOLD ENSEMBLE] "
            f"f0={len(boxes_f0)} "
            f"f1={len(boxes_f1)} "
            f"f2={len(boxes_f2)} "
            f"m={len(boxes_m)} "
            f"merged={len(boxes)}"
)
        if len(boxes) == 0:
            return boxes, scores, cls

        keep = torchvision.ops.nms(
            torch.tensor(boxes, dtype=torch.float32),
            torch.tensor(scores, dtype=torch.float32),
            0.85
        ).cpu().numpy()

        print(
            f"[ENSEMBLE] "
            f"after_nms={len(keep)}"
        )

        return (
            boxes[keep],
            scores[keep],
            cls[keep],
        )

    def predict_image(self, image_path: str | Path) -> DetectionResult:
        """
        Run inference on a single image file.

        Parameters
        ----------
        image_path : Path to the image file.

        Returns
        -------
        DetectionResult with boxes in pixel xyxy coordinates.
        """
        image_path = Path(image_path)
        img = cv2.imread(str(image_path))
        if img is None:
            log.error(f"Failed to read image: '{image_path}'")
            return DetectionResult(
                image_path=str(image_path),
                image_w=0, image_h=0,
                boxes=np.empty((0, 4), dtype=np.float32),
                scores=np.empty(0, dtype=np.float32),
            )

        img_h, img_w = img.shape[:2]

        boxes_xyxy, scores, class_ids = self._predict_raw(img)

        if len(boxes_xyxy) == 0:
            log.debug(f"  No detections in '{image_path.name}'")

            return DetectionResult(
                image_path=str(image_path),
                image_w=img_w,
                image_h=img_h,
                boxes=np.empty((0,4), dtype=np.float32),
                scores=np.empty(0, dtype=np.float32),
            )

        boxes_xyxy = clip_boxes(
            boxes_xyxy,
            img_w,
            img_h
        )

        log.debug(
            f"  '{image_path.name}': {len(boxes_xyxy)} boxes detected "
            f"(conf≥{self.conf})"
        )

        return DetectionResult(
            image_path=str(image_path),
            image_w=img_w, image_h=img_h,
            boxes=boxes_xyxy,
            scores=scores,
            class_ids=class_ids,
        )

    def predict_numpy(
        self, image: np.ndarray, source_name: str = "array"
    ) -> DetectionResult:
        """
        Run inference directly on a numpy array (HxWxC uint8, BGR or RGB).

        Parameters
        ----------
        image       : NumPy image array.
        source_name : Label used in the returned DetectionResult.image_path.
        """
        img_h, img_w = image.shape[:2]

        boxes_xyxy, scores, class_ids = self._predict_raw(image)

        if len(boxes_xyxy) == 0:
            return DetectionResult(
                image_path=source_name,
                image_w=img_w,
                image_h=img_h,
                boxes=np.empty((0,4), dtype=np.float32),
                cores=np.empty(0, dtype=np.float32),
            )

        boxes_xyxy = clip_boxes(
            boxes_xyxy,
            img_w,
            img_h
        )

        return DetectionResult(
            image_path=source_name,
            image_w=img_w,
            image_h=img_h,
            boxes=boxes_xyxy,
            scores=scores,
            class_ids=class_ids,
        )
        
        
    def predict_directory(
        self, img_dir: str | Path, extensions: Optional[set] = None
    ) -> List[DetectionResult]:
        """
        Run inference on all images in a directory.

        Parameters
        ----------
        img_dir    : Directory of page images.
        extensions : Set of image extensions to process.
        """
        if extensions is None:
            extensions = _IMG_EXTS
        img_dir = Path(img_dir)
        img_files = sorted(
            p for p in img_dir.iterdir() if p.suffix.lower() in extensions
        )
        log.info(
            f"Running detector on {len(img_files)} images in '{img_dir}'"
        )

        results: List[DetectionResult] = []
        for img_path in tqdm(img_files, desc="Detecting", unit="img"):
            results.append(self.predict_image(img_path))

        total_boxes = sum(r.num_boxes for r in results)
        log.info(
            f"Detection complete: {total_boxes} total boxes across "
            f"{len(img_files)} images"
        )
        return results

In [ ]:
# detector = DetectorInference(DET_MAIN_WEIGHTS, conf=INFER_CONF, iou=INFER_IOU, imgsz=INFER_IMGSZ)
# results  = detector.predict_directory("data/raw/images")   # List[DetectionResult]
# single   = detector.predict_image("data/raw/images/015.jpg")
print("DetectorInference (4-model ensemble + NMS@0.85) defined.")


## 6. Detection Experiments

Per-fold validation metrics from the Ultralytics training logs. The detector is
evaluated as a single class; precision is intentionally not optimised because the
competition metric ignores false positives.

**YOLO11s vs YOLO11m, by fold and image size**

| Model | Fold | imgsz | mAP@50 | Precision | Recall |
|---|:---:|:---:|---:|---:|---:|
| YOLO11s | 0 | 1280 | 0.436 | 0.786 | 0.484 |
| YOLO11s | 1 | 1280 | 0.463 | 0.721 | 0.519 |
| YOLO11s | 2 | 1280 | 0.449 | 0.807 | 0.508 |
| YOLO11s | 0 | 1536 | 0.449 | 0.806 | 0.492 |
| YOLO11s | 1 | 1536 | 0.484 | 0.726 | 0.523 |
| YOLO11s | 2 | 1536 | 0.420 | 0.769 | 0.488 |
| YOLO11m | full | 1536 | 0.466 | 0.733 | 0.528 |

Single-model mAP/recall is modest because the per-page character density is very
high (often 600+ boxes/page) and boxes are tiny. What matters for the score is
**recall at IoU ≥ 0.5 after ensembling**.

**Effect of the ensemble (measured by the competition metric over the 17 pages):**

| Configuration | Matched / GT | Recall @ IoU≥0.5 |
|---|---:|---:|
| 4-model ensemble + NMS@0.85 | 6,040 / 6,678 | **90.4 %** |

Per-page detection recall ranges from ~80 % (dense pages 011, 012) to ~97 %
(sparse pages 103–105). The `analyze_detector_iou.py` utility buckets every GT box
by its best-matching prediction IoU (`<0.3`, `0.3–0.5`, `0.5–0.7`, `≥0.7`) to
confirm that most misses are genuine non-detections rather than loose boxes.


### 6.1 Earlier single-model detector (reference)

The repository keeps `infer_detector_backup.py`, the pre-ensemble single-model implementation. It is reproduced here under a distinct name so it does not override the production `DetectorInference`. It documents the baseline before the fold ensemble was introduced.


In [ ]:
# NOTE: reuses the production DetectionResult dataclass defined above.
"""
detector/infer_detector.py
---------------------------
Run YOLO11s inference on one or more page images and return predicted boxes.

Outputs
-------
For each image the function returns:
    List[DetectionResult] where each DetectionResult has:
        image_path : str
        boxes      : np.ndarray  (N, 4) xyxy pixels
        scores     : np.ndarray  (N,)   confidence scores
        labels     : List[str]   always ["character"] * N (detector is single-class)

Usage (CLI)
-----------
    python -m src.detector.infer_detector \\
        --weights  outputs/detector/yolo11s_baseline/weights/best.pt \\
        --images   data/raw/images \\
        --out_dir  outputs/detector_preds \\
        --conf     0.03 \\
        --iou      0.50

Usage (Python API)
------------------
    detector = DetectorInference("path/to/best.pt", conf=0.03, iou=0.50)
    results = detector.predict_image("page.png")
"""

import argparse
import json
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional, Union

import cv2
import numpy as np
from tqdm import tqdm

log = get_logger(__name__)

_IMG_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

class DetectorInferenceSingleModel:
    """
    YOLO-based character detector wrapper.

    Parameters
    ----------
    weights_path : Path to trained YOLO weights (.pt).
    conf         : Confidence threshold.  Low values → high recall.
    iou          : NMS IoU threshold.
    imgsz        : Inference image size.
    device       : Torch device string ("", "cpu", "0", etc.).
    """

    def __init__(
        self,
        weights_path: str | Path,
        conf: float = 0.03,
        iou: float = 0.50,
        imgsz: int = 1280,
        device: str = "",
    ) -> None:
        try:
            from ultralytics import YOLO
        except ImportError:
            log.error("Ultralytics not installed.  pip install ultralytics")
            sys.exit(1)

        self.weights_path = Path(weights_path)
        self.conf  = conf
        self.iou   = iou
        self.imgsz = imgsz
        self.device = device

        log.info(f"Loading detector weights: '{self.weights_path}'")
        self._model = YOLO(str(self.weights_path))
        log.info(
            f"Detector ready | conf={conf} | iou={iou} | imgsz={imgsz}"
        )

    def _predict_raw(self, image):
        
        results = self._model.predict(
            source=image,
            conf=self.conf,
            iou=self.iou,
            imgsz=self.imgsz,
            device=self.device,
            max_det=10000,
            verbose=False,
            augment=True,
        )

        r = results[0]

        if r.boxes is None or len(r.boxes) == 0:
            return (
                np.empty((0,4), dtype=np.float32),
                np.empty(0, dtype=np.float32),
                np.empty(0, dtype=np.int32),
            )

        return (
            r.boxes.xyxy.cpu().numpy().astype(np.float32),
            r.boxes.conf.cpu().numpy().astype(np.float32),
            r.boxes.cls.cpu().numpy().astype(np.int32),
        )

    def predict_image(self, image_path: str | Path) -> DetectionResult:
        """
        Run inference on a single image file.

        Parameters
        ----------
        image_path : Path to the image file.

        Returns
        -------
        DetectionResult with boxes in pixel xyxy coordinates.
        """
        image_path = Path(image_path)
        img = cv2.imread(str(image_path))
        if img is None:
            log.error(f"Failed to read image: '{image_path}'")
            return DetectionResult(
                image_path=str(image_path),
                image_w=0, image_h=0,
                boxes=np.empty((0, 4), dtype=np.float32),
                scores=np.empty(0, dtype=np.float32),
            )

        img_h, img_w = img.shape[:2]

        results = self._model.predict(
            source=str(image_path),
            conf=self.conf,
            iou=self.iou,
            imgsz=self.imgsz,
            device=self.device,
            max_det=6000,
            verbose=False,
            augment=True,
        )

        # results[0].boxes.xyxy is a Tensor or ndarray of (N, 4)
        r = results[0]
        if r.boxes is None or len(r.boxes) == 0:
            log.debug(f"  No detections in '{image_path.name}'")
            return DetectionResult(
                image_path=str(image_path),
                image_w=img_w, image_h=img_h,
                boxes=np.empty((0, 4), dtype=np.float32),
                scores=np.empty(0, dtype=np.float32),
            )

        boxes_xyxy = r.boxes.xyxy.cpu().numpy().astype(np.float32)
        scores     = r.boxes.conf.cpu().numpy().astype(np.float32)
        class_ids  = r.boxes.cls.cpu().numpy().astype(np.int32)

        # Clip to image boundary (small violations possible after NMS)
        boxes_xyxy = clip_boxes(boxes_xyxy, img_w, img_h)

        log.debug(
            f"  '{image_path.name}': {len(boxes_xyxy)} boxes detected "
            f"(conf≥{self.conf})"
        )

        return DetectionResult(
            image_path=str(image_path),
            image_w=img_w, image_h=img_h,
            boxes=boxes_xyxy,
            scores=scores,
            class_ids=class_ids,
        )

    def predict_numpy(
        self, image: np.ndarray, source_name: str = "array"
    ) -> DetectionResult:
        """
        Run inference directly on a numpy array (HxWxC uint8, BGR or RGB).

        Parameters
        ----------
        image       : NumPy image array.
        source_name : Label used in the returned DetectionResult.image_path.
        """
        img_h, img_w = image.shape[:2]

        results = self._model.predict(
            source=image,
            conf=self.conf,
            iou=self.iou,
            imgsz=self.imgsz,
            device=self.device,
            max_det=6000,
            verbose=False,
            augment=True,
        )

        r = results[0]
        if r.boxes is None or len(r.boxes) == 0:
            return DetectionResult(
                image_path=source_name,
                image_w=img_w, image_h=img_h,
                boxes=np.empty((0, 4), dtype=np.float32),
                scores=np.empty(0, dtype=np.float32),
            )

        boxes_xyxy = r.boxes.xyxy.cpu().numpy().astype(np.float32)
        scores     = r.boxes.conf.cpu().numpy().astype(np.float32)
        class_ids  = r.boxes.cls.cpu().numpy().astype(np.int32)
        boxes_xyxy = clip_boxes(boxes_xyxy, img_w, img_h)

        return DetectionResult(
            image_path=source_name,
            image_w=img_w, image_h=img_h,
            boxes=boxes_xyxy,
            scores=scores,
            class_ids=class_ids,
        )

    def predict_directory(
        self, img_dir: str | Path, extensions: Optional[set] = None
    ) -> List[DetectionResult]:
        """
        Run inference on all images in a directory.

        Parameters
        ----------
        img_dir    : Directory of page images.
        extensions : Set of image extensions to process.
        """
        if extensions is None:
            extensions = _IMG_EXTS
        img_dir = Path(img_dir)
        img_files = sorted(
            p for p in img_dir.iterdir() if p.suffix.lower() in extensions
        )
        log.info(
            f"Running detector on {len(img_files)} images in '{img_dir}'"
        )

        results: List[DetectionResult] = []
        for img_path in tqdm(img_files, desc="Detecting", unit="img"):
            results.append(self.predict_image(img_path))

        total_boxes = sum(r.num_boxes for r in results)
        log.info(
            f"Detection complete: {total_boxes} total boxes across "
            f"{len(img_files)} images"
        )
        return results


## 7. Recognition Pipeline

Stage 2 classifies each detected crop into one of the 167 label strings with a
**timm ConvNeXt-Small** backbone (ImageNet-pretrained). Key design choices:

- **Every label string is its own class.** A conjunct like `U+0069+U+0069` is a
  separate class from `U+0069`; the label map is built from *sorted unique label
  strings* so the integer encoding is identical across folds, checkpoints and the
  submission.
- **Crop-level augmentation** simulates handwriting variation: rotation,
  perspective, elastic + grid distortion, brightness/contrast, Gaussian noise, and
  morphological dilation/erosion (ink bleed/thinning).
- The final configuration (`convnext_small_baseline_nocw`) **disables** class
  weighting and the weighted sampler (`use_class_weights: false`,
  `use_weighted_sampler: false`) — these were ablated and hurt or did not help.


### 7.1 Crop dataset + label map

Extracts one padded crop per box, builds the deterministic `label_map.json` (sorted unique labels → 0..166), and writes `crops_meta.csv`.


In [ ]:
"""
preprocessing/create_crop_dataset.py
--------------------------------------
Extract individual character crops from page images using the annotations CSV,
build the label→index mapping (label_map.json), and save everything to disk.

Output layout
-------------
    data/
        crops/
            <label>/
                <image_stem>_<row_idx>.png
        label_map.json   <- {label_string: class_index}
        crops_meta.csv   <- one row per crop: path, label, class_idx, image_file

Usage (CLI)
-----------
    python -m src.preprocessing.create_crop_dataset \\
        --ann_csv   data/annotations.csv \\
        --crops_dir data/crops \\
        --label_map data/label_map.json \\
        --meta_csv  data/crops_meta.csv \\
        --size      128

Usage (Python API)
------------------
    create_crop_dataset("data/annotations.csv", "data/crops", "data/label_map.json")
"""

import argparse
import json
import sys
from collections import Counter
from pathlib import Path
from typing import Dict, Optional, List

import cv2
import pandas as pd
from tqdm import tqdm

log = get_logger(__name__)

def build_label_map(labels: List[str]) -> Dict[str, int]:
    """
    Build a deterministic label → class_index mapping.

    Labels are sorted alphabetically so the mapping is stable across runs and
    machines.  This guarantees that fold files, model checkpoints, and
    submission CSVs all share the same integer encoding.

    Parameters
    ----------
    labels : Iterable of unique label strings.

    Returns
    -------
    Dict mapping each label string to a 0-based integer class index.
    """
    unique_sorted = sorted(set(labels))
    return {lbl: idx for idx, lbl in enumerate(unique_sorted)}

def create_crop_dataset(
    ann_csv: str | Path,
    crops_dir: str | Path,
    label_map_path: str | Path,
    meta_csv_path: str | Path = "data/crops_meta.csv",
    target_size: int = 128,
    pad_value: int = 255,
) -> pd.DataFrame:
    """
    Extract crops from page images and save to disk.

    Parameters
    ----------
    ann_csv        : Unified annotations CSV (from convert_labelme.py).
    crops_dir      : Root directory for saved crops.
    label_map_path : Where to write label_map.json.
    meta_csv_path  : Where to write the crops metadata CSV.
    target_size    : Output crop size (square).
    pad_value      : Padding fill colour (255 = white).

    Returns
    -------
    Metadata DataFrame (same as what is written to meta_csv_path).
    """
    ann_csv       = Path(ann_csv)
    crops_dir     = Path(crops_dir)
    label_map_path = Path(label_map_path)
    meta_csv_path  = Path(meta_csv_path)

    df = pd.read_csv(ann_csv)
    log.info(f"Loaded {len(df)} annotation rows from '{ann_csv}'")


    label_map = build_label_map(df["label"].tolist())
    label_map_path.parent.mkdir(parents=True, exist_ok=True)
    with open(label_map_path, "w", encoding="utf-8") as f:
        json.dump(label_map, f, indent=2, ensure_ascii=False)
    log.info(
        f"Label map: {len(label_map)} classes saved → '{label_map_path}'"
    )


    freq = Counter(df["label"].tolist())
    top5 = freq.most_common(5)
    bot5 = freq.most_common()[-5:]
    log.info(f"Top-5 classes (most frequent): {top5}")
    log.info(f"Bot-5 classes (least frequent): {bot5}")


    crops_dir.mkdir(parents=True, exist_ok=True)

    meta_rows: List[dict] = []
    skipped = 0

    # Group by image to avoid re-opening the same image repeatedly
    for img_file, group in tqdm(
        df.groupby("image_file"), desc="Extracting crops", unit="image"
    ):
        # Locate source image
        src_paths = group["image_path"].dropna().unique()
        if len(src_paths) == 0:
            log.warning(f"  No image_path for '{img_file}' — skipping {len(group)} boxes")
            skipped += len(group)
            continue

        src_img_path = Path(src_paths[0])
        if not src_img_path.exists():
            log.warning(f"  Image missing: '{src_img_path}' — skipping")
            skipped += len(group)
            continue

        page = cv2.imread(str(src_img_path))
        if page is None:
            log.warning(f"  cv2 failed to read '{src_img_path}' — skipping")
            skipped += len(group)
            continue

        img_stem = src_img_path.stem

        for row_idx, row in group.iterrows():
            label     = row["label"]
            class_idx = label_map[label]
            box       = [row["x1"], row["y1"], row["x2"], row["y2"]]

            crop = crop_with_padding(page, box, target_size, pad_value)

            # Save under crops_dir/<label>/<stem>_<idx>.png
            class_dir = crops_dir / label
            class_dir.mkdir(parents=True, exist_ok=True)
            crop_name = f"{img_stem}_{row_idx:05d}.png"
            crop_path = class_dir / crop_name

            cv2.imwrite(str(crop_path), crop)

            meta_rows.append(
                {
                    "crop_path":  str(crop_path),
                    "label":      label,
                    "class_idx":  class_idx,
                    "image_file": img_file,
                    "x1":         row["x1"],
                    "y1":         row["y1"],
                    "x2":         row["x2"],
                    "y2":         row["y2"],
                }
            )

    log.info(f"Extracted {len(meta_rows)} crops | Skipped: {skipped}")


    meta_df = pd.DataFrame(meta_rows)
    meta_csv_path.parent.mkdir(parents=True, exist_ok=True)
    meta_df.to_csv(meta_csv_path, index=False)
    log.info(f"Crops metadata saved → '{meta_csv_path}'")

    return meta_df

### 7.2 Recogniser folds

**Stratified 5-fold** split on `class_idx` (seed 42) so each fold preserves the global class distribution. Training uses `df.fold != k` for train and `df.fold == k` for validation. The production ensemble uses folds 0, 1, 2.


In [ ]:
"""
preprocessing/create_folds.py
------------------------------
Create stratified K-fold split files for the recogniser training.

For the **detector** the split is at image-level (handled inside
create_yolo_dataset.py).  For the **recogniser** we do crop-level stratified
splitting so that every fold has roughly equal class representation.

Output
------
    data/folds/
        fold0.csv
        fold1.csv
        ...
        fold<K-1>.csv

Each CSV has ALL crops with a column `fold` (0 to K-1) indicating which fold
the sample belongs to.  Training scripts filter on  `df[df.fold != k]` for
train and  `df[df.fold == k]` for validation.

Usage (CLI)
-----------
    python -m src.preprocessing.create_folds \\
        --meta_csv  data/crops_meta.csv \\
        --folds_dir data/folds \\
        --n_folds   5

Usage (Python API)
------------------
    create_folds("data/crops_meta.csv", "data/folds", n_folds=5)
"""

import argparse
import sys
from pathlib import Path
from typing import Optional, List

import pandas as pd
from sklearn.model_selection import StratifiedKFold

log = get_logger(__name__)

def create_folds(
    meta_csv: str | Path,
    folds_dir: str | Path,
    n_folds: int = 5,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Create stratified K-fold split files.

    Stratification is on the `class_idx` column (integer class label), which
    guarantees each fold maintains the global class distribution.

    For very rare classes (fewer samples than n_folds), StratifiedKFold will
    raise an error — we handle this by falling back to a random split for those
    classes (sklearn already handles it gracefully via n_splits <= n_samples
    adjustment with a warning).

    Parameters
    ----------
    meta_csv  : crops_meta.csv produced by create_crop_dataset.py.
    folds_dir : Directory to write fold0.csv … fold<K-1>.csv.
    n_folds   : Number of folds.
    seed      : Random seed for reproducibility.

    Returns
    -------
    Full DataFrame with a `fold` column added (0 to n_folds-1).
    """
    meta_csv  = Path(meta_csv)
    folds_dir = Path(folds_dir)
    folds_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(meta_csv)
    log.info(
        f"Loaded {len(df)} rows from '{meta_csv}' | "
        f"Classes: {df['class_idx'].nunique()}"
    )

    # Assign fold indices via StratifiedKFold
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    df["fold"] = -1

    for fold_idx, (_, val_idx) in enumerate(
        skf.split(df, df["class_idx"])
    ):
        df.loc[val_idx, "fold"] = fold_idx

    # Sanity check — no row should be unassigned
    unassigned = (df["fold"] == -1).sum()
    if unassigned > 0:
        log.warning(f"{unassigned} rows were not assigned a fold — check data.")


    for k in range(n_folds):
        fold_path = folds_dir / f"fold{k}.csv"
        df.to_csv(fold_path, index=False)  # write the full df with fold column

        n_train = (df["fold"] != k).sum()
        n_val   = (df["fold"] == k).sum()
        log.info(
            f"  fold{k}: train={n_train}, val={n_val} | "
            f"Saved → '{fold_path}'"
        )


    for k in range(n_folds):
        val_df = df[df["fold"] == k]
        class_counts = val_df["class_idx"].value_counts()
        min_c = class_counts.min()
        max_c = class_counts.max()
        log.debug(
            f"  fold{k} val class balance — min: {min_c}, max: {max_c}, "
            f"n_classes: {class_counts.shape[0]}"
        )

    log.info(f"Created {n_folds} fold files in '{folds_dir}'")
    return df

In [ ]:

#Build the recogniser datasets (run once, if annotations.csv is present)
# Mirrors the create_crop_dataset.py + create_folds.py CLI steps.
CROPS_DIR  = "data/crops"
META_CSV   = "data/crops_meta.csv"
FOLDS_DIR  = "data/folds"

if Path(ANNOTATIONS_CSV).exists():
    create_crop_dataset(
        ann_csv=ANNOTATIONS_CSV,
        crops_dir=CROPS_DIR,
        label_map_path=LABEL_MAP_PATH,
        meta_csv_path=META_CSV,
        target_size=128,      # repository used --size 128
        pad_value=255,
    )
    create_folds(meta_csv=META_CSV, folds_dir=FOLDS_DIR, n_folds=5, seed=42)
else:
    print(f"'{ANNOTATIONS_CSV}' not found — run the LabelMe conversion first.")

### 7.3 Augmentations, dataset & loaders

`build_train_transforms` / `build_val_transforms`, the `CharCropDataset`, inverse-frequency class weights, the `WeightedRandomSampler`, and `build_dataloaders` (which resolves the per-fold CSV and splits train/val).

In [ ]:
"""
recognizer/dataset.py
----------------------
PyTorch Dataset and DataLoader factory for the ConvNeXt character recogniser.

Key features:
- Loads crops from the crops_meta.csv / per-fold CSV
- Albumentations-based augmentation pipeline (train) / simple resize (val)
- WeightedRandomSampler for class imbalance
- Class-weight tensor for weighted cross-entropy

Public API
----------
    train_loader, val_loader, class_weights = build_dataloaders(cfg, fold=0)
"""

import json
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import torch
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

log = get_logger(__name__)


def build_train_transforms(cfg: dict) -> A.Compose:
    """
    Build the Albumentations training pipeline.

    Includes:
    - Geometric: Rotate, Perspective, ElasticTransform, GridDistortion
    - Intensity: RandomBrightnessContrast, GaussNoise
    - Morphological: Dilation, Erosion (via Morphological op)
    - Normalise + ToTensor

    Parameters
    ----------
    cfg : Recogniser config dict (from recognizer.yaml).
    """
    input_size = cfg.get("input_size", 128)
    ks = cfg.get("morph_kernel_size", 3)

    transforms = [
        
        A.Rotate(
            limit=cfg.get("rotate_limit", 15),
            border_mode=cv2.BORDER_CONSTANT,
            fill=255,          # white background for document crops
            p=0.6,
        ),
        A.Perspective(
            scale=(0.01, cfg.get("perspective_scale", 0.05)),
            fill=255,
            p=0.3,
        ),
        A.ElasticTransform(
            alpha=cfg.get("elastic_alpha", 1.0),
            sigma=cfg.get("elastic_sigma", 50.0),
            p=0.3,
        ),
        A.GridDistortion(
            num_steps=cfg.get("grid_distort_num_steps", 5),
            distort_limit=cfg.get("grid_distort_distort_limit", 0.3),
            p=0.3,
        ),
        
        A.RandomBrightnessContrast(
            brightness_limit=cfg.get("brightness_contrast_limit", 0.2),
            contrast_limit=cfg.get("brightness_contrast_limit", 0.2),
            p=0.5,
        ),
        A.GaussNoise(
            std_range=(
                cfg.get("gauss_noise_var_limit", [10.0, 50.0])[0] / 255.0,
                cfg.get("gauss_noise_var_limit", [10.0, 50.0])[1] / 255.0,
            ),
            p=0.4,
        ),
        
        # Simulates ink bleeding (dilation) and ink erosion/thinning (erosion)
        A.OneOf(
            [
                A.Morphological(
                    scale=(ks, ks + 2),
                    operation="dilation",
                    p=1.0,
                ),
                A.Morphological(
                    scale=(ks, ks + 2),
                    operation="erosion",
                    p=1.0,
                ),
            ],
            p=0.3,
        ),
        
        A.Resize(input_size, input_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ]

    return A.Compose(transforms)

def build_val_transforms(cfg: dict) -> A.Compose:
    """Build the minimal validation pipeline (resize + normalise only)."""
    input_size = cfg.get("input_size", 128)
    return A.Compose(
        [
            A.Resize(input_size, input_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ]
    )



class CharCropDataset(Dataset):
    """
    Dataset of character crop images for classification.

    Parameters
    ----------
    df         : DataFrame with columns: crop_path, class_idx.
    transform  : Albumentations Compose pipeline.
    label_map  : Dict[str, int] for reference (not used for lookup here).
    """

    def __init__(
        self,
        df: pd.DataFrame,
        transform: A.Compose,
        label_map: Optional[Dict[str, int]] = None,
    ) -> None:
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.label_map = label_map

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        row = self.df.iloc[idx]
        img = cv2.imread(str(row["crop_path"]))

        if img is None:
            # Return blank crop on read failure — logs a warning once
            log.warning(f"Failed to read crop: '{row['crop_path']}' — using blank.")
            img = np.full((128, 128, 3), 255, dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        augmented = self.transform(image=img)
        image_tensor: torch.Tensor = augmented["image"]
        label = int(row["class_idx"])
        return image_tensor, label



def compute_class_weights(
    class_indices: List[int], num_classes: int
) -> torch.Tensor:
    """
    Compute inverse-frequency class weights for cross-entropy loss.

    weight[c] = total_samples / (num_classes * count[c])

    Parameters
    ----------
    class_indices : List of integer class labels in the dataset.
    num_classes   : Total number of classes.

    Returns
    -------
    Float32 tensor of shape (num_classes,).
    """
    counts = torch.zeros(num_classes, dtype=torch.float32)
    for idx in class_indices:
        counts[idx] += 1

    # Avoid division by zero for classes not present in this split
    counts = torch.clamp(counts, min=1.0)
    weights = len(class_indices) / (num_classes * counts)
    return weights

def build_weighted_sampler(
    class_indices: List[int], num_classes: int
) -> WeightedRandomSampler:
    """
    Build a WeightedRandomSampler that upsamples rare classes.

    Parameters
    ----------
    class_indices : List of integer class labels (one per sample).
    num_classes   : Total number of classes.

    Returns
    -------
    WeightedRandomSampler with replacement=True.
    """
    class_weights = compute_class_weights(class_indices, num_classes)
    # Per-sample weight = weight of its class
    sample_weights = class_weights[class_indices]
    return WeightedRandomSampler(
        weights=sample_weights.tolist(),
        num_samples=len(class_indices),
        replacement=True,
    )



def build_dataloaders(
    cfg: dict,
    fold: int = 0,
) -> Tuple[DataLoader, DataLoader, Optional[torch.Tensor]]:
    """
    Build train and validation DataLoaders for a given fold.

    Parameters
    ----------
    cfg  : Recogniser config dict (from recognizer.yaml).
    fold : Which fold to use as validation set.

    Returns
    -------
    (train_loader, val_loader, class_weights_tensor)
        class_weights_tensor is None if use_class_weights=False in cfg.
    """
    folds_dir = Path(cfg["folds_dir"])

    if fold == -1:
        fold_csv = folds_dir / "fold0.csv"

        if not fold_csv.exists():
            raise FileNotFoundError(
                f"Fold file not found: '{fold_csv}'"
            )

        df = pd.read_csv(fold_csv)

        log.info(
            f"FULL TRAINING MODE | total samples: {len(df)} | "
            f"classes: {df['class_idx'].nunique()}"
        )

    else:
        fold_csv = folds_dir / f"fold{fold}.csv"

        if not fold_csv.exists():
            raise FileNotFoundError(
                f"Fold file not found: '{fold_csv}'. "
                f"Run create_folds.py first."
            )

        df = pd.read_csv(fold_csv)

        log.info(
            f"Loaded fold {fold} | total samples: {len(df)} | "
            f"classes: {df['class_idx'].nunique()}"
        )
    
    label_map_path = Path(cfg["label_map"])
    with open(label_map_path, "r", encoding="utf-8") as f:
        label_map: Dict[str, int] = json.load(f)

    num_classes = cfg.get("num_classes", -1)
    if num_classes == -1:
        num_classes = len(label_map)
    log.info(f"Number of classes: {num_classes}")

    
    if fold == -1:
        log.info("FULL TRAINING MODE ENABLED")

        train_df = df.copy()

        # Tiny validation set only to keep training code working
        val_df = df.sample(
            n=min(200, len(df)),
            random_state=42,
        ).copy()
    else:
        train_df = df[df["fold"] != fold].copy()
        val_df   = df[df["fold"] == fold].copy()

    log.info(
        f"Train samples: {len(train_df)} | Val samples: {len(val_df)}"
    )
    
    train_transform = build_train_transforms(cfg)
    val_transform   = build_val_transforms(cfg)

    train_dataset = CharCropDataset(train_df, train_transform, label_map)
    val_dataset   = CharCropDataset(val_df,   val_transform,   label_map)

    
    class_weights: Optional[torch.Tensor] = None
    if cfg.get("use_class_weights", True):
        class_weights = compute_class_weights(
            train_df["class_idx"].tolist(), num_classes
        )
        log.info(
            f"Class weights — min: {class_weights.min():.4f}, "
            f"max: {class_weights.max():.4f}"
        )

    
    sampler = None
    shuffle_train = True
    if cfg.get("use_weighted_sampler", True):
        sampler = build_weighted_sampler(
            train_df["class_idx"].tolist(), num_classes
        )
        shuffle_train = False  # cannot use shuffle with custom sampler
        log.info("WeightedRandomSampler enabled")

    
    batch_size = cfg.get("batch_size", 64)
    workers    = cfg.get("workers", 4)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=shuffle_train,
        num_workers=workers,
        pin_memory=True,
        drop_last=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=workers,
        pin_memory=True,
        drop_last=False,
    )

    log.info(
        f"DataLoaders built — "
        f"train batches: {len(train_loader)}, val batches: {len(val_loader)}"
    )
    return train_loader, val_loader, class_weights

### 7.4 Model + checkpoint utilities

`build_model` wraps a timm backbone (default `convnext_small`) with `drop_path_rate=0.1`; `save_checkpoint` / `load_checkpoint` / `load_model_weights_only` handle full and weights-only states.


In [ ]:
"""
recognizer/model.py
--------------------
ConvNeXt-Tiny classification model wrapper.

Creates a timm ConvNeXt-Tiny with the correct number of output classes,
provides checkpoint save / load helpers, and exposes a feature extraction
mode for potential future use (e.g. metric learning).

Public API
----------
    model = build_model(num_classes=167, pretrained=True)
"""

import json
from pathlib import Path
from typing import Any, Dict, Optional

import torch
import torch.nn as nn

log = get_logger(__name__)

def build_model(
    backbone: str = "convnext_small",
    num_classes: int = 167,
    pretrained: bool = True,
    drop_path_rate: float = 0.1,
) -> nn.Module:
    """
    Build a timm ConvNeXt-Tiny classification model.

    Parameters
    ----------
    backbone       : timm model name (default "convnext_tiny").
    num_classes    : Number of output classes.
    pretrained     : Use ImageNet-1k pretrained weights.
    drop_path_rate : Stochastic depth rate (regularisation).

    Returns
    -------
    PyTorch Module ready for training.
    """
    try:
        import timm
    except ImportError:
        raise ImportError("timm not installed.  pip install timm")

    log.info(
        f"Building backbone '{backbone}' | "
        f"num_classes={num_classes} | "
        f"pretrained={pretrained}"
    )

    model = timm.create_model(
        backbone,
        pretrained=pretrained,
        num_classes=num_classes,
        drop_path_rate=drop_path_rate,
    )

    # Log parameter count
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info(
        f"Model parameters — total: {total_params:,} | "
        f"trainable: {trainable_params:,}"
    )

    return model



def save_checkpoint(
    path: str | Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: Any,
    epoch: int,
    metrics: Dict[str, float],
    cfg: Dict[str, Any],
) -> None:
    """
    Save a training checkpoint.

    Saved state
    -----------
    - model state_dict
    - optimizer state_dict
    - scheduler state_dict
    - current epoch
    - metrics dict (e.g. {"val_acc": 0.92, "val_loss": 0.1})
    - config snapshot

    Parameters
    ----------
    path      : Destination .pt file path.
    model     : The model to save.
    optimizer : Optimizer being used.
    scheduler : LR scheduler (pass None if not used).
    epoch     : Current epoch index.
    metrics   : Dict of metric name → value to embed in checkpoint.
    cfg       : Config dict for reproducibility.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    state = {
        "epoch":          epoch,
        "model_state":    model.state_dict(),
        "optimizer_state":optimizer.state_dict(),
        "scheduler_state":scheduler.state_dict() if scheduler is not None else None,
        "metrics":        metrics,
        "config":         cfg,
    }
    torch.save(state, path)
    log.info(f"Checkpoint saved → '{path}' | epoch={epoch} | {metrics}")

def load_checkpoint(
    path: str | Path,
    model: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[Any] = None,
    device: str = "cpu",
) -> Dict[str, Any]:
    """
    Load a training checkpoint.

    Parameters
    ----------
    path      : Path to the .pt checkpoint file.
    model     : Model to load weights into (in-place).
    optimizer : Optional optimizer to restore state.
    scheduler : Optional scheduler to restore state.
    device    : Device to map tensors to.

    Returns
    -------
    Dict with keys: epoch, metrics, config.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Checkpoint not found: '{path}'")

    log.info(f"Loading checkpoint: '{path}'")
    state = torch.load(path, map_location=device, weights_only=False)

    model.load_state_dict(state["model_state"])

    if optimizer is not None and "optimizer_state" in state:
        optimizer.load_state_dict(state["optimizer_state"])

    if scheduler is not None and state.get("scheduler_state") is not None:
        scheduler.load_state_dict(state["scheduler_state"])

    epoch   = state.get("epoch", 0)
    metrics = state.get("metrics", {})
    cfg     = state.get("config", {})
    log.info(f"Loaded epoch={epoch} | metrics={metrics}")

    return {"epoch": epoch, "metrics": metrics, "config": cfg}

def load_model_weights_only(
    path: str | Path,
    model: nn.Module,
    device: str = "cpu",
) -> nn.Module:
    """
    Load only model weights from a checkpoint (for inference).

    Parameters
    ----------
    path   : Path to the .pt checkpoint.
    model  : Model to load weights into.
    device : Torch device string.

    Returns
    -------
    The model with loaded weights (eval mode).
    """
    state = torch.load(path, map_location=device, weights_only=False)
    # Support both raw state_dict and wrapped checkpoint
    if "model_state" in state:
        model.load_state_dict(state["model_state"])
    else:
        model.load_state_dict(state)
    model.eval()
    log.info(f"Model weights loaded from '{path}' | device={device}")
    return model

### 7.5 Recogniser training

Label-smoothing cross-entropy (`label_smoothing=0.05`), AdamW (`lr=1e-4`, `wd=1e-4`), cosine annealing to `eta_min=1e-6`, gradient clipping at norm 5.0, 30 epochs, best checkpoint by val accuracy.

In [ ]:
"""
recognizer/train_recognizer.py
--------------------------------
Train the ConvNeXt-Tiny character recogniser.

Features
--------
- Reads configs/recognizer.yaml
- 5-fold cross-validation support
- Label-smoothing cross-entropy loss
- Optional class-weighted loss for imbalanced 167-class data
- WeightedRandomSampler for balanced batches
- Cosine-annealing LR scheduler
- tqdm progress bars
- Logs train loss, val loss, val accuracy, learning rate per epoch
- Saves best checkpoint (by val accuracy)
- Resume from checkpoint

Usage (CLI)
-----------
    python -m src.recognizer.train_recognizer \\
        --config configs/recognizer.yaml \\
        --fold   0

    # Resume from checkpoint
    python -m src.recognizer.train_recognizer \\
        --config configs/recognizer.yaml \\
        --fold   0 \\
        --resume outputs/recognizer/convnext_tiny_baseline/fold0/best.pt
"""

import argparse
import json
import sys
from pathlib import Path
from typing import Dict, Optional, Tuple, List

import torch
import torch.nn as nn
import torch.optim as optim
import yaml
from tqdm import tqdm

log = get_logger(__name__)

def load_config(config_path: str | Path) -> dict:
    with open(config_path, "r") as f:
        return yaml.safe_load(f)

def resolve_device(cfg: dict) -> torch.device:
    """Select GPU if available, else CPU.  Honour cfg['device'] override."""
    device_str = cfg.get("device", "")
    if device_str == "" or device_str is None:
        device_str = "cuda" if torch.cuda.is_available() else "cpu"
    device = torch.device(device_str)
    log.info(f"Using device: {device}")
    return device

def build_criterion(
    cfg: dict,
    class_weights: Optional[torch.Tensor],
    device: torch.device,
) -> nn.Module:
    """Build cross-entropy loss, optionally weighted and with label smoothing."""
    smoothing = cfg.get("label_smoothing", 0.05)

    if cfg.get("use_class_weights", True) and class_weights is not None:
        weight = class_weights.to(device)
        log.info(f"Using weighted cross-entropy | label_smoothing={smoothing}")
    else:
        weight = None
        log.info(f"Using standard cross-entropy | label_smoothing={smoothing}")

    return nn.CrossEntropyLoss(weight=weight, label_smoothing=smoothing)

def build_optimizer(cfg: dict, model: nn.Module) -> optim.Optimizer:
    lr = cfg.get("lr", 1e-4)
    wd = cfg.get("weight_decay", 1e-4)
    log.info(f"Optimizer: AdamW | lr={lr} | weight_decay={wd}")
    return optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

def build_scheduler(
    cfg: dict, optimizer: optim.Optimizer, n_epochs: int
) -> Optional[optim.lr_scheduler._LRScheduler]:
    sched_type = cfg.get("scheduler", "cosine").lower()
    if sched_type == "cosine":
        eta_min = cfg.get("eta_min", 1e-6)
        sched = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=n_epochs, eta_min=eta_min
        )
        log.info(f"Scheduler: CosineAnnealingLR | T_max={n_epochs} | eta_min={eta_min}")
        return sched
    elif sched_type == "step":
        sched = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
        log.info("Scheduler: StepLR(step_size=10, gamma=0.5)")
        return sched
    else:
        log.info("No LR scheduler")
        return None

def run_epoch(
    model:     nn.Module,
    loader:    torch.utils.data.DataLoader,
    criterion: nn.Module,
    optimizer: Optional[optim.Optimizer],
    device:    torch.device,
    phase:     str,
) -> Tuple[float, float]:
    """
    Run a single epoch (train or validation).

    Parameters
    ----------
    model     : The ConvNeXt model.
    loader    : DataLoader for this phase.
    criterion : Loss function.
    optimizer : Optimizer (None during validation).
    device    : Torch device.
    phase     : "train" or "val".

    Returns
    -------
    (avg_loss, accuracy) for this epoch.
    """
    is_train = phase == "train"
    model.train(is_train)

    total_loss = 0.0
    correct    = 0
    total      = 0

    desc = f"  [{phase.upper()}]"
    with tqdm(loader, desc=desc, leave=False, unit="batch") as pbar:
        for images, labels in pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.set_grad_enabled(is_train):
                logits = model(images)
                loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                # Gradient clipping for stability
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

            batch_size  = images.size(0)
            total_loss += loss.item() * batch_size
            preds       = logits.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += batch_size

            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                acc=f"{correct/total:.4f}",
            )

    avg_loss = total_loss / max(total, 1)
    accuracy = correct    / max(total, 1)
    return avg_loss, accuracy

def train_recognizer(
    config_path: str | Path,
    fold:        int = 0,
    resume:      Optional[str] = None,
) -> None:
    """
    Full training loop for the ConvNeXt recogniser.

    Parameters
    ----------
    config_path : Path to recognizer.yaml.
    fold        : Fold index (0 to n_folds-1).
    resume      : Optional path to a checkpoint to resume from.
    """
    cfg = load_config(config_path)
    seed_everything(cfg.get("seed", 42))

  
    label_map_path = Path(cfg["label_map"])
    with open(label_map_path, "r", encoding="utf-8") as f:
        label_map: Dict[str, int] = json.load(f)
    num_classes = len(label_map)
    cfg["num_classes"] = num_classes
    log.info(f"Number of classes: {num_classes}")


    train_loader, val_loader, class_weights = build_dataloaders(cfg, fold=fold)


    device = resolve_device(cfg)


    model = build_model(
        backbone=cfg.get("backbone", "convnext_tiny"),
        num_classes=num_classes,
        pretrained=cfg.get("pretrained", True),
    ).to(device)

    criterion = build_criterion(cfg, class_weights, device)
    optimizer = build_optimizer(cfg, model)
    n_epochs  = cfg.get("epochs", 50)
    scheduler = build_scheduler(cfg, optimizer, n_epochs)

    run_name = cfg.get("run_name", "convnext_tiny_baseline")
    project  = Path(cfg.get("project", "outputs/recognizer"))
    out_dir  = project / f"{run_name}_fold{fold}"
    out_dir.mkdir(parents=True, exist_ok=True)
    log.info(f"Checkpoints will be saved to '{out_dir}'")

   
    start_epoch = 0
    best_val_acc = 0.0

    if resume:
        ckpt_info = load_checkpoint(resume, model, optimizer, scheduler, device)
        start_epoch  = ckpt_info["epoch"] + 1
        best_val_acc = ckpt_info["metrics"].get("val_acc", 0.0)
        log.info(
            f"Resuming from epoch {start_epoch} | best_val_acc={best_val_acc:.4f}"
        )

    
    log.info(
        f"=== Starting recogniser training | fold={fold} | "
        f"epochs={n_epochs} | classes={num_classes} ==="
    )

    history: List[Dict[str, float]] = []

    for epoch in range(start_epoch, n_epochs):
        current_lr = optimizer.param_groups[0]["lr"]
        log.info(
            f"Epoch [{epoch+1}/{n_epochs}] | LR: {current_lr:.6f}"
        )

        
        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, optimizer, device, "train"
        )
        
        val_loss, val_acc = run_epoch(
            model, val_loader, criterion, None, device, "val"
        )

        if scheduler is not None:
            scheduler.step()

        log.info(
            f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        metrics = {
            "train_loss": train_loss,
            "train_acc":  train_acc,
            "val_loss":   val_loss,
            "val_acc":    val_acc,
            "lr":         current_lr,
        }
        history.append({"epoch": epoch, **metrics})

        
        save_checkpoint(
            path=out_dir / "last.pt",
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            epoch=epoch,
            metrics=metrics,
            cfg=cfg,
        )

       
        if cfg.get("save_best", True) and val_acc > best_val_acc:
            best_val_acc = val_acc
            save_checkpoint(
                path=out_dir / "best.pt",
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                metrics=metrics,
                cfg=cfg,
            )
            log.info(f"  *** New best val_acc: {best_val_acc:.4f} → 'best.pt' saved ***")

    
    import json as _json
    hist_path = out_dir / "history.json"
    with open(hist_path, "w") as f:
        _json.dump(history, f, indent=2)
    log.info(f"Training history saved → '{hist_path}'")

    log.info(
        f"=== Recogniser Training Complete | fold={fold} | "
        f"best_val_acc={best_val_acc:.4f} ==="
    )

In [ ]:

# Train the three recogniser folds
# Mirrors:  python -m src.recognizer.train_recognizer --config configs/recognizer.yaml --fold {k}
# The repository run_name is "convnext_small_baseline_nocw" → outputs land in
#   outputs/recognizer/convnext_small_baseline_nocw_fold{k}/best.pt
#
# for k in (0, 1, 2):
#     train_recognizer(config_path="configs/recognizer.yaml", fold=k)
print("Recogniser training entry point ready (call train_recognizer(...) to launch).")

### 7.6 Recogniser inference

`RecognizerInference` loads a single backbone, applies the resize+ImageNet-normalise transform, and exposes `predict_batch` (label, confidence) plus `predict_batch_logits` (raw logits, used by the ensembles). Default backbone is `convnext_small`, `input_size=224`.

In [ ]:
"""
recognizer/infer_recognizer.py
--------------------------------
Run ConvNeXt-Tiny inference on character crop images.

Supports:
- Single crop image
- List of numpy arrays (for pipeline integration)
- Batch inference with tqdm progress bar

Public API
----------
    rec = RecognizerInference("outputs/recognizer/.../best.pt",
                              label_map_path="data/label_map.json")
    label = rec.predict_image("crop.png")
    labels = rec.predict_batch(crops_numpy_list)
"""

import argparse
import json
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import albumentations as A
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

log = get_logger(__name__)

def _build_infer_transform(input_size: int = 224) -> A.Compose:
    """Minimal inference pipeline: resize + ImageNet normalise."""
    return A.Compose(
        [
            A.Resize(input_size, input_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ]
    )

class RecognizerInference:
    """
    ConvNeXt character recogniser inference wrapper.

    Parameters
    ----------
    weights_path    : Path to the .pt checkpoint (full or weights-only).
    label_map_path  : Path to label_map.json (label_string → class_idx).
    backbone        : timm backbone name (must match training config).
    input_size      : Expected crop size (128 for default config).
    device          : Torch device string ("", "cpu", "0", etc.).
    batch_size      : Internal batch size for batch inference.
    """

    def __init__(
        self,
        weights_path: str | Path,
        label_map_path: str | Path,
        backbone: str = "convnext_small",
        input_size: int = 224,
        device: str = "",
        batch_size: int = 128,
    ) -> None:
    
        if device == "" or device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = torch.device(device)
        self.batch_size = batch_size


        with open(label_map_path, "r", encoding="utf-8") as f:
            label_map: Dict[str, int] = json.load(f)
        # Invert: class_idx → label_string
        self.idx_to_label: Dict[int, str] = {v: k for k, v in label_map.items()}
        num_classes = len(label_map)
        log.info(f"Label map loaded: {num_classes} classes")


        model = build_model(backbone, num_classes=num_classes, pretrained=False)
        self.model = load_model_weights_only(
            weights_path, model, device=str(self.device)
        ).to(self.device)
        self.model.eval()


        self.transform = _build_infer_transform(input_size)
        log.info(
            f"Recogniser ready | backbone={backbone} | "
            f"classes={num_classes} | device={self.device}"
        )

    def _preprocess(self, image_bgr: np.ndarray) -> torch.Tensor:
        """Preprocess a single BGR image to a normalised tensor."""
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        augmented = self.transform(image=image_rgb)
        return augmented["image"]   # shape (3, H, W)

    @torch.no_grad()
    def predict_image(self, image_path: str | Path) -> Tuple[str, float]:
        """
        Predict the label for a single image file.

        Parameters
        ----------
        image_path : Path to the crop PNG/JPG.

        Returns
        -------
        (predicted_label_string, confidence_score)
        """
        img = cv2.imread(str(image_path))
        if img is None:
            log.warning(f"Cannot read '{image_path}' — returning empty label")
            return "", 0.0
        return self._predict_single(img)

    @torch.no_grad()
    def _predict_single(
        self, image_bgr: np.ndarray
    ) -> Tuple[str, float]:
        """Internal: predict from a single BGR numpy array."""
        tensor = self._preprocess(image_bgr).unsqueeze(0).to(self.device)
        logits = self.model(tensor)
        probs  = F.softmax(logits, dim=1)
        conf, pred_idx = probs.max(dim=1)
        label = self.idx_to_label.get(int(pred_idx.item()), "UNKNOWN")
        return label, float(conf.item())

    @torch.no_grad()
    def predict_batch(
        self,
        images_bgr: List[np.ndarray],
        show_progress: bool = False,
    ) -> List[Tuple[str, float]]:
        """
        Predict labels for a list of BGR numpy arrays.

        Processes in mini-batches of self.batch_size for memory efficiency.

        Parameters
        ----------
        images_bgr    : List of HxWxC uint8 BGR images.
        show_progress : Whether to show a tqdm bar.

        Returns
        -------
        List of (label_string, confidence) tuples, same order as input.
        """
        if not images_bgr:
            return []

        results: List[Tuple[str, float]] = []
        n = len(images_bgr)

        # Pre-process all images into tensors
        tensors: List[torch.Tensor] = []
        for img in images_bgr:
            if img is None or img.size == 0:
                tensors.append(
                    torch.zeros(3, 224, 224, dtype=torch.float32)
                )
            else:
                tensors.append(self._preprocess(img))

        # Batch inference
        iterator = range(0, n, self.batch_size)
        if show_progress:
            iterator = tqdm(iterator, desc="Recognizing", unit="batch")

        for start in iterator:
            end   = min(start + self.batch_size, n)
            batch = torch.stack(tensors[start:end]).to(self.device)
            logits = self.model(batch)
            probs  = F.softmax(logits, dim=1)
            confs, preds = probs.max(dim=1)

            for pred_idx, conf in zip(preds.cpu().tolist(), confs.cpu().tolist()):
                label = self.idx_to_label.get(int(pred_idx), "UNKNOWN")
                results.append((label, float(conf)))

        return results

    @torch.no_grad()
    def predict_batch_logits(
        self,
        images_bgr: List[np.ndarray],
    ) -> torch.Tensor:

        if not images_bgr:
            return torch.empty(0)

        tensors = []

        for img in images_bgr:
            if img is None or img.size == 0:
                tensors.append(
                    torch.zeros(3, 224, 224, dtype=torch.float32)
                )
            else:
                tensors.append(self._preprocess(img))

        all_logits = []

        for start in range(0, len(tensors), self.batch_size):

            end = min(start + self.batch_size, len(tensors))

            batch = torch.stack(
                tensors[start:end]
            ).to(self.device)

            logits = self.model(batch)

            all_logits.append(
                logits.cpu()
            )

        return torch.cat(all_logits, dim=0)


## 8. Recognition Experiments

**Cross-validation accuracy (final config: `convnext_small`, no class weights,
no weighted sampler, 30 epochs):**

| Fold | Backbone | Best val accuracy |
|:---:|---|---:|
| 0 | ConvNeXt-Small | 0.835 |
| 1 | ConvNeXt-Small | 0.833 |
| 2 | ConvNeXt-Small | 0.823 |
| **mean** | | **≈ 0.830** |

**Ablations (single fold, early runs).** Toggling class weighting and the
weighted sampler was explored. The `nocw` (no-class-weights) configuration was
kept because the weighted variants did not improve held-out accuracy and made
training noisier on the long tail. "Full-training" runs (`fold = -1`) report very
high "val" accuracy (up to 1.00) but that split's validation set is a 200-sample
*subset of the training data*, so those numbers are not comparable to true CV and
were not used for selection.

**End-to-end recognition (competition metric, matched boxes only):** unicode
accuracy on matched boxes ranges ~80 %–96 % per page, ≈ **92 %** weighted overall.
The `analyze_recognizer_accuracy.py` utility measures this directly by matching
each GT box to its best detection (IoU ≥ 0.5) and checking the predicted label.


### 8.1 Tiny+Small recogniser ensemble (reference)
An alternative two-backbone ensemble (ConvNeXt-Tiny + ConvNeXt-Small, 50/50 logit blend). Preserved for completeness; the production submission uses the 3-fold Small ensemble instead.


In [ ]:
from pathlib import Path
from typing import List, Tuple

import numpy as np
import torch
import torch.nn.functional as F

class EnsembleRecognizerInference:

    def __init__(
        self,
        tiny_weights,
        small_weights,
        label_map_path,
        device="",
        batch_size=128,
        input_size=224,
    ):

        self.tiny = RecognizerInference(
            weights_path=tiny_weights,
            label_map_path=label_map_path,
            backbone="convnext_tiny",
            input_size=input_size,
            device=device,
            batch_size=batch_size,
        )

        self.small = RecognizerInference(
            weights_path=small_weights,
            label_map_path=label_map_path,
            backbone="convnext_small",
            input_size=input_size,
            device=device,
            batch_size=batch_size,
        )

        self.idx_to_label = self.tiny.idx_to_label

    @torch.no_grad()
    def predict_batch(
        self,
        images_bgr,
        show_progress=False,
    ):

        logits_tiny = self.tiny.predict_batch_logits(images_bgr)
        logits_small = self.small.predict_batch_logits(images_bgr)

        print(
            f"[ENSEMBLE] Tiny logits shape: {tuple(logits_tiny.shape)} | "
            f"Small logits shape: {tuple(logits_small.shape)}"
        )
        
        tiny_preds = logits_tiny.argmax(dim=1)
        small_preds = logits_small.argmax(dim=1)

        disagree = (tiny_preds != small_preds).sum().item()

        print(
            f"[ENSEMBLE] Model disagreements: "
            f"{disagree}/{len(tiny_preds)}"
        )

        logits = (
            0.50 * logits_tiny +
            0.50 * logits_small
        )

        probs = F.softmax(logits, dim=1)

        confs, preds = probs.max(dim=1)

        results = []

        for pred_idx, conf in zip(
            preds.tolist(),
            confs.tolist()
        ):
            label = self.idx_to_label.get(
                int(pred_idx),
                "UNKNOWN"
            )

            results.append(
                (
                    label,
                    float(conf)
                )
            )

        return results


## 9. Ensemble Strategy

Two independent ensembles are used, one per stage.

**Detector ensemble (recall-oriented union).** Four YOLO models each predict with
test-time augmentation at a very low confidence (0.01). All boxes are concatenated
and passed through a single NMS at a **high** IoU of 0.85. Because NMS only
suppresses boxes overlapping more than 0.85, complementary detections survive and
the union recall exceeds any single model's. This trades precision for recall —
exactly what the metric rewards.

**Recogniser ensemble (variance reduction).** `FoldEnsembleRecognizer` runs the
three fold checkpoints, **averages their logits** (`(l0+l1+l2)/3`), then applies
softmax and argmax. Averaging in logit space is equivalent to a geometric mean of
per-model probabilities and is more stable on rare classes than majority voting.
The label/confidence decoding is byte-for-byte identical to the single-model path
(`idx_to_label.get(idx, "UNKNOWN")`).


### 9.1 Fold-ensemble recogniser (production) 


In [ ]:
import torch
import torch.nn.functional as F

class FoldEnsembleRecognizer:

    def __init__(
        self,
        fold0_weights,
        fold1_weights,
        fold2_weights,
        label_map_path,
        device="",
        batch_size=128,
        input_size=224,
    ):

        self.fold0 = RecognizerInference(
            weights_path=fold0_weights,
            label_map_path=label_map_path,
            backbone="convnext_small",
            input_size=input_size,
            device=device,
            batch_size=batch_size,
        )

        self.fold1 = RecognizerInference(
            weights_path=fold1_weights,
            label_map_path=label_map_path,
            backbone="convnext_small",
            input_size=input_size,
            device=device,
            batch_size=batch_size,
        )

        self.fold2 = RecognizerInference(
            weights_path=fold2_weights,
            label_map_path=label_map_path,
            backbone="convnext_small",
            input_size=input_size,
            device=device,
            batch_size=batch_size,
        )

        self.idx_to_label = self.fold0.idx_to_label

    @torch.no_grad()
    def predict_batch(
        self,
        images_bgr,
        show_progress=False,
    ):

        logits0 = self.fold0.predict_batch_logits(images_bgr)
        logits1 = self.fold1.predict_batch_logits(images_bgr)
        logits2 = self.fold2.predict_batch_logits(images_bgr)

        pred0 = logits0.argmax(dim=1)
        pred1 = logits1.argmax(dim=1)
        pred2 = logits2.argmax(dim=1)

        d01 = (pred0 != pred1).sum().item()
        d02 = (pred0 != pred2).sum().item()

        #print(
        #    f"[FOLD ENSEMBLE] "
        #    f"F0/F1 disagreements={d01} "
        #    f"F0/F2 disagreements={d02}"
        #)

        logits = (
            logits0 +
            logits1 +
            logits2
        ) / 3.0

        probs = F.softmax(logits, dim=1)

        confs, preds = probs.max(dim=1)

        results = []

        for pred_idx, conf in zip(
            preds.tolist(),
            confs.tolist()
        ):
            label = self.idx_to_label.get(
                int(pred_idx),
                "UNKNOWN"
            )

            results.append(
                (
                    label,
                    float(conf)
                )
            )

        return results


## 10. Submission Generation

This section wires both stages together and writes the final CSV.

**The exact submission format (preserved verbatim).** The output has two columns,
`page_id` and `predictions`. `page_id` is the image filename stem. `predictions`
is a single string built as:

```
[ "script":<0|1>, {"unicode_value":"<label>","bbox":[x1,y1,x2,y2]}, ... ]
```

where each `bbox` coordinate is formatted to 4 decimals, and `script` is a
page-level majority flag: **1 if more predicted labels start with `U+09`
(Bengali) than not, else 0**. This is the literal string the repository emits —
note it is not strict JSON (the leading `"script":N` has no enclosing object); it
is reproduced exactly so scoring behaviour is unchanged.

The competition metric is included first so the pipeline can be scored locally.


### 10.1 Competition metric

Hungarian matching on `1 − IoU`, `IoU ≥ 0.50` to accept a match, and the `0.20/0.40/0.40` script/unicode/iou weighting. Reproduced exactly, including the original `evaluate_page` return-shape behaviour.


In [ ]:
"""
metrics/competition_metric.py
------------------------------
Exact reproduction of the COMSYS Hackathon 7 evaluation metric.

Scoring formula
---------------
    char_score = 0.20 * script_score
               + 0.40 * unicode_score
               + 0.40 * iou_score

Where:
- script_score  : 1 if the predicted label belongs to the same script family,
                  else 0.  (If script families are not available, treated as
                  unicode_score > 0.)
- unicode_score : 1 if predicted label == ground-truth label, else 0.
- iou_score     : IoU of the matched predicted box and the GT box.

Assignment strategy
-------------------
Hungarian matching (scipy.optimize.linear_sum_assignment) over the IoU
matrix between GT and predicted boxes.  A match is only valid when IoU ≥ 0.5.

Missing boxes (GT boxes without a matched prediction) are penalised by
contributing 0 to all three sub-scores.
Extra predicted boxes (predictions without a matched GT) are ignored.

This matches the competition's stated methodology.

Usage (Python API)
------------------

    page_score = evaluate_page(
        gt_boxes=np.array([[x1,y1,x2,y2], ...]),
        gt_labels=["U+0065", ...],
        pred_boxes=np.array([[x1,y1,x2,y2], ...]),
        pred_labels=["U+0065", ...],
    )

Usage (CLI)
-----------
    python -m src.metrics.competition_metric \\
        --gt_csv   data/annotations.csv \\
        --pred_csv outputs/predictions.csv
"""

import argparse
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from collections import Counter

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

log = get_logger(__name__)


IOU_THRESHOLD = 0.50

WEIGHT_SCRIPT  = 0.20
WEIGHT_UNICODE = 0.40
WEIGHT_IOU     = 0.40


# A minimal heuristic: two labels share a script if they have the same Unicode
# block prefix (first codepoint range).  This is a reasonable proxy; teams
# with access to the competition's official script mapping should replace this.

def _unicode_codepoints(label: str) -> List[int]:
    """Parse 'U+0065' or 'U+0069+U+0069' → list of int codepoints."""
    parts = label.split("+")
    codepoints = []
    for part in parts:
        part = part.strip()
        if part.startswith("U"):
            try:
                codepoints.append(int(part[2:], 16))
            except ValueError:
                pass
    return codepoints

def _script_family(label: str) -> str:
    """
    Assign a coarse script family based on the first codepoint.

    Unicode block boundaries (simplified):
        0x0000 – 0x007F : Latin Basic
        0x0080 – 0x00FF : Latin Extended-A/B
        0x0100 – 0x024F : Latin Extended
        0x0370 – 0x03FF : Greek
        0x0400 – 0x04FF : Cyrillic
        0x0600 – 0x06FF : Arabic
        0x0900 – 0x097F : Devanagari
        0x0980 – 0x09FF : Bengali
        0x0A00 – 0x0A7F : Gurmukhi
        0x0A80 – 0x0AFF : Gujarati
        0x0B00 – 0x0B7F : Oriya
        0x0B80 – 0x0BFF : Tamil
        0x0C00 – 0x0C7F : Telugu
        0x0C80 – 0x0CFF : Kannada
        0x0D00 – 0x0D7F : Malayalam
        0x0E00 – 0x0E7F : Thai
        0x4E00 – 0x9FFF : CJK
        ... everything else → "other"
    """
    cps = _unicode_codepoints(label)
    if not cps:
        return "unknown"
    cp = cps[0]

    if   0x0000 <= cp <= 0x024F: return "latin"
    elif 0x0370 <= cp <= 0x03FF: return "greek"
    elif 0x0400 <= cp <= 0x04FF: return "cyrillic"
    elif 0x0600 <= cp <= 0x06FF: return "arabic"
    elif 0x0900 <= cp <= 0x097F: return "devanagari"
    elif 0x0980 <= cp <= 0x09FF: return "bengali"
    elif 0x0A00 <= cp <= 0x0A7F: return "gurmukhi"
    elif 0x0A80 <= cp <= 0x0AFF: return "gujarati"
    elif 0x0B00 <= cp <= 0x0B7F: return "oriya"
    elif 0x0B80 <= cp <= 0x0BFF: return "tamil"
    elif 0x0C00 <= cp <= 0x0C7F: return "telugu"
    elif 0x0C80 <= cp <= 0x0CFF: return "kannada"
    elif 0x0D00 <= cp <= 0x0D7F: return "malayalam"
    elif 0x0E00 <= cp <= 0x0E7F: return "thai"
    elif 0x4E00 <= cp <= 0x9FFF: return "cjk"
    else:                         return "other"



@dataclass
class PageScore:
    """Score breakdown for a single page."""
    n_gt:          int    = 0
    n_pred:        int    = 0
    n_matched:     int    = 0
    sum_iou:       float  = 0.0
    sum_unicode:   float  = 0.0
    sum_script:    float  = 0.0
    char_score:    float  = 0.0 # final weighted score averaged over n_gt
    detection_recall: float = 0.0
    recognition_acc:  float = 0.0
    
    
    def to_dict(self) -> dict:
        return {
            "n_gt":        self.n_gt,
            "n_pred":      self.n_pred,
            "n_matched":   self.n_matched,
            "sum_iou":     self.sum_iou,
            "sum_unicode": self.sum_unicode,
            "sum_script":  self.sum_script,
            "char_score":  self.char_score,
        }



def evaluate_page(
    gt_boxes:    np.ndarray,
    gt_labels:   List[str],
    pred_boxes:  np.ndarray,
    pred_labels: List[str],
    iou_thresh:  float = IOU_THRESHOLD,
) -> PageScore:
    """
    Evaluate one page image.

    Algorithm
    ---------
    1. Compute pairwise IoU matrix (M x N) where M = #GT, N = #pred.
    2. Run Hungarian assignment on (1 - IoU) as cost.  We maximise IoU.
    3. Accept matches where IoU ≥ iou_thresh.
    4. For each accepted match:
         iou_score     = iou value
         unicode_score = 1 if pred_label == gt_label else 0
         script_score  = 1 if script(pred) == script(gt) else 0
         char_score_i  = WEIGHT_SCRIPT  * script_score
                       + WEIGHT_UNICODE * unicode_score
                       + WEIGHT_IOU    * iou_score
    5. Unmatched GT boxes contribute char_score_i = 0.
    6. Final page char_score = mean(char_score_i) over all GT boxes.

    Parameters
    ----------
    gt_boxes    : (M, 4) float32, xyxy pixels.
    gt_labels   : List of M ground-truth label strings.
    pred_boxes  : (N, 4) float32, xyxy pixels.
    pred_labels : List of N predicted label strings.
    iou_thresh  : Minimum IoU to accept a match.

    Returns
    -------
    PageScore dataclass.
    """
    ps = PageScore(n_gt=len(gt_boxes), n_pred=len(pred_boxes))
    matched_pairs=[]

    if ps.n_gt == 0:
        ps.char_score = 1.0  # nothing to penalise
        return ps

    if ps.n_pred == 0:
        # All GT boxes unmatched → zero score
        ps.char_score = 0.0
        return ps

    
    iou_mat = iou_matrix(
        np.asarray(gt_boxes, dtype=np.float32),
        np.asarray(pred_boxes, dtype=np.float32),
    )   # shape (M, N)

    
    cost_mat = 1.0 - iou_mat
    gt_idx, pred_idx = linear_sum_assignment(cost_mat)

    
    char_scores_per_gt = np.zeros(ps.n_gt, dtype=np.float64)

    for gi, pi in zip(gt_idx, pred_idx):
        iou_val = float(iou_mat[gi, pi])
        if iou_val < iou_thresh:
            # Match is below threshold — treat as unmatched (score 0)
            continue

        ps.n_matched   += 1
        ps.sum_iou     += iou_val

        gt_lbl   = gt_labels[gi]
        pred_lbl = pred_labels[pi]
        
        matched_pairs.append((gt_lbl, pred_lbl))

        unicode_s = 1.0 if pred_lbl == gt_lbl else 0.0
        script_s  = 1.0 if _script_family(pred_lbl) == _script_family(gt_lbl) else 0.0

        ps.sum_unicode += unicode_s
        ps.sum_script  += script_s

        char_score_i = (
            WEIGHT_SCRIPT  * script_s +
            WEIGHT_UNICODE * unicode_s +
            WEIGHT_IOU     * iou_val
        )
        char_scores_per_gt[gi] = char_score_i

    # Mean over all GT (unmatched GT contribute 0)
    ps.char_score = float(char_scores_per_gt.mean())
    
    ps.detection_recall = (
        ps.n_matched / ps.n_gt
        if ps.n_gt > 0
        else 0.0
    )

    ps.recognition_acc = (
        ps.sum_unicode / ps.n_matched
        if ps.n_matched > 0
        else 0.0
    )

    return ps, matched_pairs



def evaluate_dataset(
    gt_df:   pd.DataFrame,
    pred_df: pd.DataFrame,
    iou_thresh: float = IOU_THRESHOLD,
) -> Tuple[float, Dict[str, PageScore]]:
    """
    Evaluate across all pages in the dataset.

    Parameters
    ----------
    gt_df   : DataFrame with columns: image_file, x1, y1, x2, y2, label
    pred_df : DataFrame with columns: image_file, x1, y1, x2, y2, pred_label

    Returns
    -------
    (mean_char_score, {image_file: PageScore})
    """
    all_images = set(gt_df["image_file"].unique())
    page_scores: Dict[str, PageScore] = {}
    
    confusions = Counter()

    for img_file in sorted(all_images):
        gt_rows   = gt_df[gt_df["image_file"] == img_file]
        pred_rows = pred_df[pred_df["image_file"] == img_file] \
                    if img_file in pred_df["image_file"].values else pd.DataFrame()

        gt_boxes   = gt_rows[["x1","y1","x2","y2"]].values.astype(np.float32)
        gt_labels  = gt_rows["label"].tolist()

        if pred_rows.empty:
            pred_boxes  = np.empty((0, 4), dtype=np.float32)
            pred_labels = []
        else:
            pred_boxes  = pred_rows[["x1","y1","x2","y2"]].values.astype(np.float32)
            pred_labels = pred_rows["pred_label"].tolist()

        ps,pairs = evaluate_page(gt_boxes, gt_labels, pred_boxes, pred_labels, iou_thresh)
        page_scores[img_file] = ps
        
        for gt_lbl, pred_lbl in pairs:

            if gt_lbl != pred_lbl:
                confusions[(gt_lbl, pred_lbl)] += 1

        log.info(
            f"  [{img_file}] "
            f"GT={ps.n_gt} | "
            f"Pred={ps.n_pred} | "
            f"Matched={ps.n_matched} | "
            f"Recall={100*ps.detection_recall:.2f}% | "
            f"RecAcc={100*ps.recognition_acc:.2f}% | "
            f"Score={100*ps.char_score:.2f}"
        )
        
        mean_iou = ps.sum_iou / ps.n_matched
        print(f"MeanIoU={100*mean_iou:.2f}%")

    if not page_scores:
        return 0.0, {}

    total_gt = 0
    total_char_score = 0.0

    for ps in page_scores.values():
        total_gt += ps.n_gt
        total_char_score += ps.char_score * ps.n_gt

    mean_score = (
        total_char_score / total_gt
        if total_gt > 0
        else 0.0
    )

    log.info(
        f"Dataset mean char_score: {mean_score:.4f} "
        f"(weighted over {total_gt} GT chars)"
    )
    
    print("\nTOP CONFUSIONS\n")

    for (gt_lbl, pred_lbl), count in confusions.most_common(50):

        print(
            f"{gt_lbl:15s} -> "
            f"{pred_lbl:15s} : "
            f"{count}"
        )

    return mean_score, page_scores

### 10.2 Two-stage page predictor

`PagePredictor` runs detection → crop extraction (128 px, white pad) → recognition. `from_configs` builds the **production** ensembles: the YOLO11m main detector (whose `DetectorInference` also loads the three YOLO11s folds) and the 3-fold ConvNeXt-Small recogniser. The hardcoded checkpoint paths are preserved verbatim.


In [ ]:
"""
pipeline/predict_page.py
-------------------------
Full two-stage inference for a single page image.

Pipeline
--------
1. Run YOLO detector   → bounding boxes in pixel xyxy coordinates
2. Extract crops       → aspect-ratio-preserving pad + resize to 128x128
3. Run ConvNeXt recogniser → predicted label strings + confidence scores
4. Return structured results

This module is the core integration point between Stage 1 and Stage 2.
Both generate_submission.py and the local evaluation notebook use this.

Public API
----------

    predictor = PagePredictor.from_configs(
        detector_weights="outputs/detector/.../best.pt",
        recognizer_weights="outputs/recognizer/.../best.pt",
        label_map_path="data/label_map.json",
    )
    prediction = predictor.predict("data/raw/images/page_01.jpg")
    print(prediction.labels)   # List[str]
    print(prediction.boxes)    # (N, 4) xyxy
"""

import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional, Tuple

import cv2
import numpy as np

log = get_logger(__name__)


@dataclass
class PagePrediction:
    """
    Complete predictions for one page image.

    Attributes
    ----------
    image_path    : Absolute path to the source image.
    image_w       : Image width in pixels.
    image_h       : Image height in pixels.
    boxes         : (N, 4) float32, xyxy pixel coordinates.
    det_scores    : (N,) float32, detector confidence per box.
    labels        : List of N predicted label strings.
    rec_scores    : List of N recogniser confidence scores.
    """
    image_path: str
    image_w:    int
    image_h:    int
    boxes:      np.ndarray = field(default_factory=lambda: np.empty((0, 4)))
    det_scores: np.ndarray = field(default_factory=lambda: np.empty(0))
    labels:     List[str]  = field(default_factory=list)
    rec_scores: List[float]= field(default_factory=list)

    @property
    def num_predictions(self) -> int:
        return len(self.labels)

    def to_rows(self) -> List[dict]:
        """Convert to a list of flat dicts (one per predicted character)."""
        image_file = Path(self.image_path).name
        rows = []
        for i, (box, det_s, label, rec_s) in enumerate(
            zip(self.boxes, self.det_scores, self.labels, self.rec_scores)
        ):
            rows.append(
                {
                    "image_file":  image_file,
                    "box_idx":     i,
                    "x1":          float(box[0]),
                    "y1":          float(box[1]),
                    "x2":          float(box[2]),
                    "y2":          float(box[3]),
                    "det_score":   float(det_s),
                    "pred_label":  label,
                    "rec_score":   float(rec_s),
                }
            )
        return rows



class PagePredictor:
    """
    Two-stage page predictor:
        detector → crops → recogniser.

    Parameters
    ----------
    detector    : Initialised DetectorInference instance.
    recognizer  : Initialised RecognizerInference instance.
    crop_size   : Side length to resize each crop to before recognition.
    pad_value   : Pad fill colour (255 = white background).
    """

    def __init__(
        self,
        detector:   DetectorInference,
        recognizer: RecognizerInference,
        crop_size:  int = 224,
        pad_value:  int = 255,
    ) -> None:
        self.detector   = detector
        self.recognizer = recognizer
        self.crop_size  = crop_size
        self.pad_value  = pad_value

    @classmethod
    def from_configs(
        cls,
        detector_weights:    str | Path,
        recognizer_weights:  str | Path,
        label_map_path:      str | Path,
        detector_conf:       float = 0.01,
        detector_iou:        float = 0.60,
        detector_imgsz:      int   = 1536,
        recognizer_backbone: str   = "convnext_tiny",
        recognizer_input:    int   = 224,
        device:              str   = "",
        rec_batch_size:      int   = 128,
        crop_size:           int   = 128,
    ) -> "PagePredictor":
        """
        Factory method: build detector + recogniser from weight paths.

        Parameters
        ----------
        detector_weights   : Path to YOLO best.pt.
        recognizer_weights : Path to ConvNeXt best.pt.
        label_map_path     : Path to label_map.json.
        ... (other kwargs map directly to DetectorInference / RecognizerInference)
        """
        det = DetectorInference(
            weights_path=r"runs\detect\outputs\detector\yolo11m_baseline-2\weights\best.pt",
            conf=detector_conf,
            iou=detector_iou,
            imgsz=detector_imgsz,
            device=device,
        )
        rec = FoldEnsembleRecognizer(
            fold0_weights="outputs/recognizer/convnext_small_baseline_nocw_fold0/best.pt",
            fold1_weights="outputs/recognizer/convnext_small_baseline_nocw_fold1/best.pt",
            fold2_weights="outputs/recognizer/convnext_small_baseline_nocw_fold2/best.pt",
            label_map_path=label_map_path,
            device=device,
            batch_size=rec_batch_size,
            input_size=recognizer_input,
        )
        return cls(det, rec, crop_size=crop_size)

    def predict(self, image_path: str | Path) -> PagePrediction:
        """
        Run the full pipeline on a single page image.

        Parameters
        ----------
        image_path : Path to the page image file.

        Returns
        -------
        PagePrediction with all detected + classified characters.
        """
        image_path = Path(image_path)

    
        det_result = self.detector.predict_image(image_path)
        log.info(
            f"[{image_path.name}] Stage 1 — {det_result.num_boxes} boxes detected"
        )

        if det_result.num_boxes == 0:
            return PagePrediction(
                image_path=str(image_path),
                image_w=det_result.image_w,
                image_h=det_result.image_h,
            )

   
        page_img = cv2.imread(str(image_path))
        if page_img is None:
            log.error(f"Failed to read image for cropping: '{image_path}'")
            return PagePrediction(
                image_path=str(image_path),
                image_w=det_result.image_w,
                image_h=det_result.image_h,
            )

        # Extract all crops from the page
        crops: List[np.ndarray] = []
        for box in det_result.boxes:
            crop = crop_with_padding(page_img, box, self.crop_size, self.pad_value)
            crops.append(crop)

        # Batch recognition
        rec_results: List[Tuple[str, float]] = self.recognizer.predict_batch(
            crops, show_progress=False
        )

        labels     = [r[0] for r in rec_results]
        rec_scores = [r[1] for r in rec_results]

        log.info(
            f"[{image_path.name}] Stage 2 — {len(labels)} labels predicted"
        )

        return PagePrediction(
            image_path  = str(image_path),
            image_w     = det_result.image_w,
            image_h     = det_result.image_h,
            boxes       = det_result.boxes,
            det_scores  = det_result.scores,
            labels      = labels,
            rec_scores  = rec_scores,
        )

    def predict_numpy(
        self, image: np.ndarray, source_name: str = "page"
    ) -> PagePrediction:
        """
        Run the full pipeline on a numpy array (BGR uint8).

        Parameters
        ----------
        image       : HxWxC BGR image array.
        source_name : Identifier used in PagePrediction.image_path.
        """
        img_h, img_w = image.shape[:2]

        det_result = self.detector.predict_numpy(image, source_name)
        log.info(
            f"[{source_name}] Stage 1 — {det_result.num_boxes} boxes detected"
        )

        if det_result.num_boxes == 0:
            return PagePrediction(
                image_path=source_name,
                image_w=img_w, image_h=img_h,
            )

        crops = [
            crop_with_padding(image, box, self.crop_size, self.pad_value)
            for box in det_result.boxes
        ]

        rec_results = self.recognizer.predict_batch(crops, show_progress=False)
        labels     = [r[0] for r in rec_results]
        rec_scores = [r[1] for r in rec_results]

        return PagePrediction(
            image_path=source_name,
            image_w=img_w, image_h=img_h,
            boxes      = det_result.boxes,
            det_scores = det_result.scores,
            labels     = labels,
            rec_scores = rec_scores,
        )

### 10.3 Submission CSV generation 

Runs the predictor over every test image and assembles the `page_id,predictions` CSV with the exact string format described above.


In [ ]:
"""
pipeline/generate_submission.py
---------------------------------
Generate a competition submission CSV by running the full pipeline on all
test images.

Behaviour
---------
1. Load sample_submission.csv to infer the required columns automatically.
2. Run detector + recogniser on every image in the test directory.
3. Map predicted class indices back to unicode label strings.
4. Write submission.csv in the exact column order of sample_submission.csv.

Column inference
----------------
The script inspects sample_submission.csv and tries to detect:
  - image identifier column (contains "image" or "file" in header)
  - box coordinate columns (x1/y1/x2/y2 or similar)
  - label column (contains "label" or "unicode" in header)

If the auto-detection fails, the user can pass --id_col, --label_col, etc.

Usage (CLI)
-----------
    python -m src.pipeline.generate_submission \\
        --test_dir       data/test/images \\
        --detector_weights  outputs/detector/.../best.pt \\
        --recognizer_weights outputs/recognizer/.../best.pt \\
        --label_map      data/label_map.json \\
        --sample_sub     data/sample_submission.csv \\
        --out_csv        outputs/submission.csv \\
        --conf           0.03 \\
        --iou            0.50
"""

import argparse
import sys
from pathlib import Path
from typing import List, Optional

import pandas as pd
from tqdm import tqdm

log = get_logger(__name__)

_IMG_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}



def _find_col(columns: List[str], keywords: List[str]) -> Optional[str]:
    """Return the first column whose name contains any keyword (case-insensitive)."""
    for col in columns:
        for kw in keywords:
            if kw in col.lower():
                return col
    return None

def infer_submission_columns(sample_df: pd.DataFrame) -> dict:
    """
    Auto-detect submission CSV columns from the sample file.

    Returns a dict with keys: image_col, label_col, x1_col, y1_col, x2_col, y2_col.
    Missing keys map to None (caller must handle).
    """
    cols = list(sample_df.columns)
    return {
        "image_col": _find_col(cols, ["image", "file", "page", "img"]),
        "label_col": _find_col(cols, ["label", "unicode", "char", "text"]),
        "x1_col":    _find_col(cols, ["x1", "xmin", "left"]),
        "y1_col":    _find_col(cols, ["y1", "ymin", "top"]),
        "x2_col":    _find_col(cols, ["x2", "xmax", "right"]),
        "y2_col":    _find_col(cols, ["y2", "ymax", "bottom"]),
    }



def generate_submission(
    test_dir:            str | Path,
    detector_weights:    str | Path,
    recognizer_weights:  str | Path,
    label_map_path:      str | Path,
    sample_sub_path:     str | Path,
    out_csv:             str | Path,
    conf:                float = 0.01,
    iou:                 float = 0.60,
    imgsz:               int   = 1280,
    backbone:            str   = "convnext_tiny",
    input_size:          int   = 224,
    device:              str   = "",
    rec_batch_size:      int   = 128,
    # Column overrides (use if auto-detection fails)
    image_col_override:  Optional[str] = None,
    label_col_override:  Optional[str] = None,
    x1_col_override:     Optional[str] = None,
    y1_col_override:     Optional[str] = None,
    x2_col_override:     Optional[str] = None,
    y2_col_override:     Optional[str] = None,
) -> pd.DataFrame:
    """
    Run the full pipeline on test images and write a submission CSV.

    Parameters
    ----------
    test_dir             : Directory of test page images.
    detector_weights     : Path to YOLO best.pt.
    recognizer_weights   : Path to ConvNeXt best.pt.
    label_map_path       : Path to label_map.json.
    sample_sub_path      : Path to sample_submission.csv.
    out_csv              : Output path for submission.csv.
    conf                 : Detector confidence threshold.
    iou                  : Detector NMS IoU threshold.
    ...
    *_col_override       : Force specific column names (bypass auto-detection).

    Returns
    -------
    DataFrame written to out_csv.
    """
    test_dir = Path(test_dir)
    out_csv  = Path(out_csv)


    sample_df = pd.read_csv(sample_sub_path)
    log.info(
        f"Sample submission columns: {list(sample_df.columns)}"
    )
    col_map = infer_submission_columns(sample_df)

    # Apply overrides
    if image_col_override: col_map["image_col"] = image_col_override
    if label_col_override: col_map["label_col"] = label_col_override
    if x1_col_override:    col_map["x1_col"]    = x1_col_override
    if y1_col_override:    col_map["y1_col"]    = y1_col_override
    if x2_col_override:    col_map["x2_col"]    = x2_col_override
    if y2_col_override:    col_map["y2_col"]    = y2_col_override

    log.info(f"Column mapping: {col_map}")


    img_files = sorted(
        p for p in test_dir.iterdir() if p.suffix.lower() in _IMG_EXTS
    )
    if not img_files:
        log.error(f"No images found in '{test_dir}'")
        sys.exit(1)
    log.info(f"Found {len(img_files)} test images in '{test_dir}'")


    predictor = PagePredictor.from_configs(
        detector_weights=detector_weights,
        recognizer_weights=recognizer_weights,
        label_map_path=label_map_path,
        detector_conf=conf,
        detector_iou=iou,
        detector_imgsz=imgsz,
        recognizer_backbone=backbone,
        recognizer_input=input_size,
        device=device,
        rec_batch_size=rec_batch_size,
    )


    import json

    submission_rows = []

    for img_path in tqdm(img_files, desc="Processing test images", unit="page"):
        pred = predictor.predict(img_path)

        latin_count = 0
        bengali_count = 0

        for row in pred.to_rows():

            label = str(row["pred_label"])

            if label.startswith("U+09"):
                bengali_count += 1
            else:
                latin_count += 1

        page_script = 1 if bengali_count > latin_count else 0
        
        parts = [f'"script":{page_script}']
        
        for row in pred.to_rows():
            parts.append(
                '{{"unicode_value":"{}","bbox":[{:.4f},{:.4f},{:.4f},{:.4f}]}}'.format(
                    row["pred_label"],
                    float(row["x1"]),
                    float(row["y1"]),
                    float(row["x2"]),
                    float(row["y2"]),
                )
            )

        prediction_string = "[" + ", ".join(parts) + "]"

        submission_rows.append({
            "page_id": Path(img_path).stem,
            "predictions": prediction_string
        })

    submission_df = pd.DataFrame(
        submission_rows,
        columns=["page_id", "predictions"]
    )

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    submission_df.to_csv(out_csv, index=False)
    log.info(
        f"Submission saved → '{out_csv}' | "
        f"{len(submission_df)} rows | columns: {list(submission_df.columns)}"
    )
    return submission_df

In [ ]:

# Generate the submission (uncomment to run; needs test images + all checkpoints)
# Mirrors:  python -m src.pipeline.generate_submission \
#               --test_dir data/test/images \
#               --detector_weights   <ignored: from_configs uses DET_MAIN_WEIGHTS> \
#               --recognizer_weights <ignored: from_configs uses the fold ensemble> \
#               --label_map data/label_map.json \
#               --sample_sub data/sample_submission.csv \
#               --out_csv outputs/submission.csv \
#               --conf 0.01 --iou 0.60
#
# submission_df = generate_submission(
#     test_dir="data/test/images",
#     detector_weights=DET_MAIN_WEIGHTS,        # path is accepted but from_configs overrides it
#     recognizer_weights=REC_FOLD0_WEIGHTS,     # likewise overridden by the fold ensemble
#     label_map_path=LABEL_MAP_PATH,
#     sample_sub_path="data/sample_submission.csv",
#     out_csv="outputs/submission.csv",
#     conf=INFER_CONF,
#     iou=INFER_IOU,
#     imgsz=INFER_IMGSZ,
#     backbone="convnext_small",
#     input_size=REC_INPUT,
#     device="",            # "" → auto (cuda if available)
#     rec_batch_size=128,
# )
# submission_df.head()
print("generate_submission(...) ready.")

In [ ]:

# Local scoring against ground truth (optional)
# Build a prediction CSV in the metric's schema (image_file, x1,y1,x2,y2, pred_label)
# from PagePredictor outputs, then call evaluate_dataset.
#
# predictor = PagePredictor.from_configs(
#     detector_weights=DET_MAIN_WEIGHTS,
#     recognizer_weights=REC_FOLD0_WEIGHTS,
#     label_map_path=LABEL_MAP_PATH,
#     detector_conf=INFER_CONF, detector_iou=INFER_IOU, detector_imgsz=INFER_IMGSZ,
#     recognizer_input=REC_INPUT, device="",
# )
# rows = []
# for img_path in sorted(Path("data/raw/images").iterdir()):
#     pred = predictor.predict(img_path)
#     for r in pred.to_rows():
#         rows.append({"image_file": r["image_file"], "x1": r["x1"], "y1": r["y1"],
#                      "x2": r["x2"], "y2": r["y2"], "pred_label": r["pred_label"]})
# pred_df = pd.DataFrame(rows)
# gt_df = pd.read_csv(ANNOTATIONS_CSV)
# mean_score, page_scores = evaluate_dataset(gt_df, pred_df)
# print("char_score:", mean_score)
print("Local evaluation snippet ready.")


## 11. Results

**Local competition metric (full pipeline over the 17 annotated pages):**

| Metric | Value |
|---|---:|
| Mean `char_score` (weighted over 6,678 GT chars) | **0.7768** |
| Detection recall @ IoU ≥ 0.5 (matched 6,040 / 6,678) | 90.4 % |
| Unicode accuracy on matched boxes (weighted) | ≈ 92 % |

**Per-page breakdown (selected):**

| Page | GT | Pred | Matched | Recall | RecAcc | char_score |
|---|---:|---:|---:|---:|---:|---:|
| 002 | 626 | 3371 | 534 | 85.3 % | 93.1 % | 72.6 |
| 003 | 497 | 2362 | 474 | 95.4 % | 93.9 % | 83.4 |
| 009 | 667 | 3251 | 643 | 96.4 % | 96.4 % | 84.7 |
| 011 | 493 | 2603 | 394 | 79.9 % | 84.3 % | 64.7 |
| 015 | 690 | 2990 | 627 | 90.9 % | 93.1 % | 77.5 |
| 104 | 254 | 1346 | 247 | 97.2 % | 96.0 % | 86.9 |

**Reading the numbers.**

- The very large `Pred` counts (often 4–6× the GT count) are expected and harmless:
  the metric ignores extra boxes, and the low-confidence, high-recall ensemble
  deliberately over-generates candidates.
- The score is **gated by detection recall**, not recognition. On pages where the
  ensemble recovers ≥ 95 % of boxes the page score approaches the ~0.85 ceiling
  implied by the 0.40-IoU + 0.40-unicode + 0.20-script weighting; the weakest pages
  (011, 012) are the densest, where small/overlapping characters are missed.
- **Leaderboard discussion.** Because the public metric rewards recall and exact
  unicode, the highest-leverage improvements are (a) closing the remaining ~10 %
  detection gap on dense pages and (b) reducing confusions among visually similar
  Bengali conjuncts (surfaced by the metric's `TOP CONFUSIONS` report). Both move
  the score more than further detector-precision tuning, which the metric ignores.


## Thankyou